[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/10_deep_learning/_assets/03_finetuning_lora.ipynb)

# Notebook 3 · Fine-tuning generativo con LoRA

- Unidad 10 de *deep learning*, clase 3. **Stable Diffusion 1.5 aprende el estilo «cara de anime»** con LoRA.
- Pregunta: **¿cuánto del modelo hay que mover, y con qué precisión, para que aprenda un estilo nuevo?**
- Se compara fine-tuning completo (no cabe en la GPU), LoRA, QLoRA y varios rangos $r$.

**Cómo correrlo**

- En Colab: «Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU** → Guardar», y luego «Ejecutar todo».
- La primera celda de código fija `MODO`:

| `MODO` | Qué corre | Tiempo en una T4 |
|---|---|---|
| `"corto"` (por omisión) | Lo esencial (✔) | ≈ 30 min |
| `"largo"` | Además, las variantes marcadas ➕ | ≈ 75 min |

- En `MODO = "corto"`, una celda ➕ imprime «(solo en MODO largo)» y el notebook sigue.

**Test, una sola vez.** Todas las decisiones se toman con validation. La loss de test se calcula **solo al final (§13), una vez**, con el modelo ya elegido.

**Sobre las caras.** El dataset reúne caras de personajes de anime cuyos derechos son de sus autores. Se usan solo para estudiar el método. **Se entrena con un único prompt genérico, `"an anime face"`, sin nombres de personajes**: el modelo aprende un estilo, no identidades. El notebook guardado muestra caras *generadas* y, como mucho, una tira de 8 caras reales.

**Mapa**

| § | Sección | Modo |
|---|---|---|
| 01 | Instalar y preparar | ✔ |
| 02 | Datos: caras, partición train / validation / test | ✔ |
| 03 | Cómo aprende un modelo de difusión | ✔ |
| 04 | LoRA y QLoRA en matemáticas | ✔ |
| 05 | La caja de herramientas | ✔ |
| 06 | Antes de entrenar: el modelo base | ✔ |
| 07 | Los cuatro casos: completo, LoRA, QLoRA, rangos | ✔ / ➕ rangos |
| 08 | Dónde poner LoRA | ➕ (corto: solo $\lVert\Delta W\rVert$ por capa) |
| 09 | Learning rate y schedule | ➕ |
| 10 | Cuánto cambia el modelo | ✔ |
| 11 | Memorización | ✔ |
| 12 | ¿Olvidó lo que sabía? | ✔ |
| 13 | Cierre: test una vez, tabla final, receta, tarea | ✔ |

## 01 · Instalar y preparar

- **Elige el modo en la primera línea de código.** `MODO = "corto"` tarda ≈ 30 min en una T4; `MODO = "largo"` agrega las variantes marcadas ➕ (≈ 75 min).
- La celda instala solo lo que falta (`diffusers`, `peft`, `accelerate`, `bitsandbytes`). No actualiza lo que Colab ya trae.

| Paquete | Para qué |
|---|---|
| `diffusers` | Stable Diffusion: UNet, VAE, schedulers |
| `transformers` | el text encoder de CLIP |
| `peft` | los adapters LoRA |
| `bitsandbytes` | pesos en 4 bits (QLoRA); solo con GPU |

In [ ]:
MODO = "corto"   # "corto" (≈ 30 min en T4) o "largo" (≈ 75 min, todo lo ➕)
LARGO = MODO == "largo"

# Instala solo lo que falta; no actualiza lo que Colab ya trae.
import importlib.util
import subprocess
import sys

import torch

FALTAN = [p for p in ["diffusers", "peft", "accelerate", "transformers"]
          if importlib.util.find_spec(p) is None]
# bitsandbytes (4 bits, QLoRA) solo funciona con GPU de NVIDIA.
if torch.cuda.is_available() and importlib.util.find_spec("bitsandbytes") is None:
    FALTAN.append("bitsandbytes")
if FALTAN:
    print("Instalando:", ", ".join(FALTAN))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *FALTAN],
                   check=True)
else:
    print("No falta nada que instalar.")
print("MODO =", MODO, "· LARGO =", LARGO)

- **Necesitas GPU.** Si aparece el aviso, elige T4 GPU en «Cambiar tipo de entorno» y ejecuta todo de nuevo.
- `DTYPE` es fp16 en la GPU: la mitad de memoria que fp32. La T4 no tiene bf16.
- `RAPIDO=1` es solo para la prueba local: usa un Stable Diffusion diminuto en CPU. Sus imágenes son ruido.
- Guarda las versiones que imprime la celda si quieres reproducir una corrida.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import diffusers
import peft
import transformers
import huggingface_hub

# RAPIDO=1: prueba local en CPU con un Stable Diffusion diminuto (segundos).
RAPIDO = os.environ.get("RAPIDO") == "1"
HAY_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAY_GPU and not RAPIDO else "cpu")
# fp16 en la GPU (la T4 no tiene bf16); fp32 en CPU.
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

if not HAY_GPU and not RAPIDO:
    print("=" * 72)
    print("  ¡AVISO! ESTE ENTORNO NO TIENE GPU.")
    print("  Stable Diffusion en CPU tardaría horas. En Colab:")
    print("    Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")
    print("    → Guardar, y luego Entorno de ejecución → Ejecutar todo.")
    print("=" * 72)

for nombre, mod in [("torch", torch), ("diffusers", diffusers),
                    ("peft", peft), ("transformers", transformers),
                    ("huggingface_hub", huggingface_hub)]:
    print(f"{nombre:16s}{mod.__version__}")
if importlib.util.find_spec("bitsandbytes") is not None:
    import bitsandbytes
    print(f"{'bitsandbytes':16s}{bitsandbytes.__version__}")
else:
    print(f"{'bitsandbytes':16s}— (sin GPU: QLoRA se saltará)")
print("DEVICE =", DEVICE, "· DTYPE =", DTYPE, "· RAPIDO =", RAPIDO)
if HAY_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} · {props.total_memory / 2**30:.1f} GiB")

- **Un solo caption para todas las caras:** `PROMPT = "an anime face"`. Sin nombres de personajes, el modelo aprende un estilo y no identidades.
- `P` junta los tamaños del modo: personajes, caras por personaje, resolución, batch, pasos de entrenamiento y cada cuántos pasos se mide validation.
- `semilla(s)` fija el azar. Misma semilla ⇒ mismo ruido inicial ⇒ imágenes comparables entre métodos.
- Colores: azul es train, naranja es validation y verde es test.

In [ ]:
# El modelo base y el modelo de CLIP que mide las imágenes.
if RAPIDO:
    MODELO = "hf-internal-testing/tiny-stable-diffusion-pipe"
    MODELO_CLIP = "hf-internal-testing/tiny-random-CLIPModel"
else:
    MODELO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    MODELO_CLIP = "openai/clip-vit-base-patch32"
DATASET = "ComputerVisionAnimeProject/AnimeFaceColorization"
PROMPT = "an anime face"      # el mismo caption para todas las caras

# Tamaños por modo. pasos = actualizaciones del UNet por corrida.
if RAPIDO:
    P = dict(n_personajes=4, n_por_personaje=4, res=32, batch=2, pasos=3,
             pasos_control=4, val_cada=1, n_val_ruido=2, semillas=[0, 1])
elif LARGO:
    P = dict(n_personajes=40, n_por_personaje=30, res=512, batch=4,
             pasos=400, pasos_control=800, val_cada=50, n_val_ruido=16,
             semillas=[0, 1, 2, 3, 4, 5])
else:
    P = dict(n_personajes=20, n_por_personaje=20, res=512, batch=4,
             pasos=150, pasos_control=300, val_cada=30, n_val_ruido=8,
             semillas=[0, 1, 2, 3])
display(pd.Series(P, name="valor").to_frame())

# Un color fijo por conjunto, igual que en el notebook 02.
COLORES = {"train": "#1f77b4", "validation": "#ff7f0e", "test": "#2ca02c"}


def semilla(s: int):
    """Fija las semillas de random, NumPy y PyTorch (CPU y GPU)."""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)        # también siembra la GPU


semilla(0)
plt.style.use("default")
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 80, "savefig.dpi": 80,
                     "axes.spines.top": False, "axes.spines.right": False})
try:                            # en Colab: figuras en JPEG, el notebook pesa menos
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 80})
except Exception:
    pass

## 02 · Datos: caras de anime

- **Qué vamos a enseñarle al modelo:** el estilo «cara de anime». Las caras vienen de la colección *moeimouto*, compilada en el dataset `ComputerVisionAnimeProject/AnimeFaceColorization` (CC-BY-4.0).
- Cada carpeta `reference/NNN_<personaje>` tiene ≈ 64 caras de 252 px. Elegimos personajes y caras con semilla; el resto de las carpetas no se usa.
- Todas las caras llevan el mismo caption, `"an anime face"`.

| Paso | Qué hace | Por qué |
|---|---|---|
| Manifiesto | lista exacta de archivos, elegidos con semilla | misma muestra en cada corrida |
| Deduplicar | hash de la cara reducida a 16×16 | una cara repetida en train y test premia memorizar |
| Partir | 80 % train, 10 % validation, 10 % test **dentro de cada personaje** | los tres conjuntos ven los mismos personajes |
| Reescalar | 252 → 512 px, bicúbico | SD 1.5 trabaja a 512 px |

- **Ojo con el reescalado:** agrandar 252 → 512 suaviza los bordes. El modelo también aprende ese suavizado.
- Validation sirve para elegir. Test se usa **una sola vez**, en el cierre.

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor

from huggingface_hub import HfApi, hf_hub_download

# Versión del dataset fijada por commit: mismas caras en cada corrida.
REV_DATOS = "e8deff76ecee132a5eead1b03941789ba60098fa"
API = HfApi()


def listar(carpeta):
    """Rutas dentro de una carpeta del dataset (la API pagina sola)."""
    return [x.path for x in API.list_repo_tree(
        DATASET, repo_type="dataset", path_in_repo=carpeta, revision=REV_DATOS)]


# Solo las carpetas reference/NNN_<personaje>; AnimeDiffusion_* no son caras.
CARPETAS = sorted(r for r in listar("reference")
                  if re.match(r"^\d{3}_", r.split("/")[-1]))
azar = random.Random(0)
ELEGIDOS = sorted(azar.sample(CARPETAS, P["n_personajes"]))
with ThreadPoolExecutor(16) as pool:
    ARCHIVOS = dict(zip(ELEGIDOS, pool.map(listar, ELEGIDOS)))

# Manifiesto explícito: N caras por personaje, elegidas con semilla.
# Pedimos 4 de más por si la deduplicación descarta alguna.
MANIFIESTO = {}
for carpeta, rutas in ARCHIVOS.items():
    pngs = sorted(r for r in rutas if r.endswith(".png"))
    MANIFIESTO[carpeta] = azar.sample(pngs, min(len(pngs),
                                                P["n_por_personaje"] + 4))
print(f"{len(CARPETAS)} personajes en el dataset; usamos {len(ELEGIDOS)}:")
print(", ".join(c.split("/")[-1][4:] for c in ELEGIDOS))

In [ ]:
import hashlib

from PIL import Image


def bajar(ruta):
    """Descarga una cara (≈ 100 KB) y la abre como RGB."""
    local = hf_hub_download(DATASET, ruta, repo_type="dataset",
                            revision=REV_DATOS)
    return Image.open(local).convert("RGB")


def huella(img):
    """Hash de la imagen reducida a 16×16: detecta duplicados casi exactos."""
    return hashlib.md5(img.resize((16, 16), Image.BILINEAR).tobytes()).hexdigest()


todas = [r for rutas in MANIFIESTO.values() for r in rutas]
with ThreadPoolExecutor(16) as pool:
    IMAGENES = dict(zip(todas, pool.map(bajar, todas)))

# Deduplicar ANTES de partir: una cara repetida en train y en test
# haría que test premie memorizar.
vistas, DATOS = set(), {"train": [], "validation": [], "test": []}
filas, duplicadas = [], 0
for carpeta, rutas in MANIFIESTO.items():
    caras = []
    for r in rutas:
        h = huella(IMAGENES[r])
        if h in vistas:
            duplicadas += 1
            continue
        vistas.add(h)
        caras.append(IMAGENES[r])
    caras = caras[:P["n_por_personaje"]]
    # Partición por imagen DENTRO de cada personaje: 80 / 10 / 10.
    n_val = n_test = max(1, round(0.1 * len(caras)))
    partes = {"train": caras[n_val + n_test:], "validation": caras[:n_val],
              "test": caras[n_val:n_val + n_test]}
    for nombre, lista in partes.items():
        # 252 px → P["res"] con interpolación bicúbica.
        DATOS[nombre] += [im.resize((P["res"], P["res"]), Image.BICUBIC)
                          for im in lista]
    filas.append({"personaje": carpeta.split("/")[-1][4:],
                  **{k: len(v) for k, v in partes.items()}})

random.Random(0).shuffle(DATOS["train"])    # mezcla los personajes
CONTROL = DATOS["train"][:2 if RAPIDO else 16]
conteo = pd.DataFrame(filas).set_index("personaje")
conteo.loc["total"] = conteo.sum()
display(conteo)
print(f"duplicadas descartadas: {duplicadas} · CONTROL: {len(CONTROL)} caras"
      f" · tamaño original {IMAGENES[todas[0]].size} → {DATOS['train'][0].size}")

- Ocho caras reales de train, reducidas a 128 px. Es la única vez que el notebook muestra caras reales; el resto son generadas.
- `CONTROL` son 16 caras de train. En la sección de memorización entrenamos con ellas muchos pasos a propósito.

In [ ]:
# Tira de 8 caras reales de train, a 128 px: la única vez que se muestran.
tira = DATOS["train"][:8]
fig, ejes = plt.subplots(1, len(tira), figsize=(1.6 * len(tira), 1.9))
for ax, img in zip(np.atleast_1d(ejes), tira):
    ax.imshow(img.resize((128, 128), Image.BICUBIC))
    ax.axis("off")
fig.suptitle(f"Caras reales de train (de {len(DATOS['train'])})", fontsize=10)
fig.tight_layout()
plt.show()

## 03 · Cómo aprende un modelo de difusión

Stable Diffusion tiene tres piezas. **Solo el UNet se entrena** en este notebook.

| Pieza | Entrada → salida | En este notebook |
|---|---|---|
| VAE | imagen 512×512×3 ↔ latent 64×64×4 | congelado; se usa una vez |
| Text encoder (CLIP) | prompt → embedding $c$ de 77 tokens | congelado; se usa una vez |
| UNet $\varepsilon_\theta$ | $(x_t, t, c)$ → ruido estimado | **se entrena** (completo, LoRA o QLoRA) |

**Ensuciar** (forward, DDPM). Con $x_0$ el latent de una cara y $\bar\alpha_t$ una constante que baja de ≈ 1 a ≈ 0:

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \varepsilon\sim\mathcal N(0,I),\quad t\sim\mathcal U\{1,\dots,T\},\ T=1000.$$

**Aprender** (ε-prediction). El UNet adivina qué ruido se le sumó:

$$\mathcal L(\theta)=\mathbb E_{x_0,t,\varepsilon}\Big[\big\lVert\varepsilon-\varepsilon_\theta(x_t,t,c)\big\rVert_2^2\Big].$$

- Es un **MSE**: regresión, como en la unidad 9. La MSE promedia sobre todos los elementos del latent.
- Para generar, se parte de ruido puro y se le resta, paso a paso, el ruido que estima el UNet.
- Aquí $x_0$ es el latent **ya multiplicado** por `scaling_factor` (0.18215), para que tenga varianza ≈ 1.

- **Precomputar una vez.** El VAE y el text encoder no se entrenan, así que sus salidas no cambian. Calculamos todos los latents y el embedding del prompt ahora.
- Después los sacamos de la GPU: cada paso de entrenamiento solo corre el UNet. Por eso cada corrida tarda minutos y no horas.
- Un latent tiene 48 veces menos números que la imagen. El UNet trabaja en ese espacio chico.

In [ ]:
import gc

from diffusers import AutoencoderKL, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

VAE = AutoencoderKL.from_pretrained(MODELO, subfolder="vae",
                                    torch_dtype=DTYPE).to(DEVICE).eval()
TOKENIZER = CLIPTokenizer.from_pretrained(MODELO, subfolder="tokenizer")
TEXTO = CLIPTextModel.from_pretrained(MODELO, subfolder="text_encoder",
                                      torch_dtype=DTYPE).to(DEVICE).eval()
ESCALA_VAE = VAE.config.scaling_factor          # 0.18215 en SD 1.5


def a_tensor(imgs):
    """Lista de PIL → tensor (N, 3, H, W) en [-1, 1], como lo espera el VAE."""
    x = torch.stack([torch.from_numpy(np.array(im)) for im in imgs])
    return x.permute(0, 3, 1, 2).float().div(127.5).sub(1)


@torch.no_grad()
def codificar(imgs, lote=8):
    """Latents x_0 = escala · media del VAE, por lotes, guardados en CPU."""
    salida = []
    for i in range(0, len(imgs), lote):
        x = a_tensor(imgs[i:i + lote]).to(DEVICE, DTYPE)
        z = VAE.encode(x).latent_dist.mode()     # la media: determinista
        salida.append((z * ESCALA_VAE).float().cpu())
    return torch.cat(salida)


LATENTS = {k: codificar(v) for k, v in DATOS.items()}

with torch.no_grad():
    ids = TOKENIZER(PROMPT, padding="max_length", truncation=True,
                    max_length=TOKENIZER.model_max_length,
                    return_tensors="pt").input_ids.to(DEVICE)
    EMB = TEXTO(ids)[0].float().cpu()            # (1, 77, D)

for k, z in LATENTS.items():
    print(f"LATENTS[{k!r}]: {tuple(z.shape)}")
print("EMB:", tuple(EMB.shape))
pix, lat = 3 * P["res"] ** 2, LATENTS["train"][0].numel()
print(f"una imagen: {pix:,} números → un latent: {lat:,} ({pix / lat:.0f}× menos)")

- **La loss de validation necesita ruido fijo.** Si cada medición sorteara $t$ y $\varepsilon$ nuevos, la curva temblaría por puro azar.
- Fijamos $K$ pares $(t,\varepsilon)$ con semilla 0, con $t$ espaciados de 0 a $T-1$. Todas las corridas se miden con los mismos pares.
- Cómo leer la tabla: con $t$ chico casi todo es señal; con $t$ cerca de $T$ casi todo es ruido.

In [ ]:
# Pares (t, ε) FIJOS para medir la loss de validation: misma semilla,
# mismos t ⇒ la curva de validation no tiembla por el azar del ruido.
RUIDO = DDPMScheduler.from_pretrained(MODELO, subfolder="scheduler")
T = RUIDO.config.num_train_timesteps
K = P["n_val_ruido"]
gen = torch.Generator().manual_seed(0)
t_val = torch.linspace(0, T - 1, K).round().long()      # espaciados en [0, T-1]
eps_val = torch.randn((K, *LATENTS["train"].shape[1:]), generator=gen)
RUIDO_VAL = (t_val, eps_val)
abar = RUIDO.alphas_cumprod
display(pd.DataFrame({"t": t_val.tolist(),
                      "√ᾱ_t (señal)": abar[t_val].sqrt().numpy().round(3),
                      "√(1-ᾱ_t) (ruido)": (1 - abar[t_val]).sqrt().numpy().round(3)}
                     ).set_index("t").T)

- Arriba: una cara de train y los 4 canales de su latent. El latent todavía conserva la silueta de la cara.
- Abajo: $x_t$ decodificado para tres valores de $t$. Con $t$ chico se ve la cara; con $t$ grande, casi solo ruido.
- Al final, la celda libera el VAE y el text encoder de la GPU.

In [ ]:
# Una cara de train: su latent (4 canales) y x_t para tres valores de t.
x0 = LATENTS["train"][:1]
eps = torch.randn(x0.shape, generator=torch.Generator().manual_seed(1))
TS = [100, 500, 900] if T > 900 else [T // 10, T // 2, 9 * T // 10]
xts = [RUIDO.add_noise(x0, eps, torch.tensor([t])) for t in TS]
with torch.no_grad():   # el VAE regresa cada latent a píxeles para verlo
    vistas = [VAE.decode((z / ESCALA_VAE).to(DEVICE, DTYPE)).sample
              for z in [x0, *xts]]
vistas = [((v[0].float().cpu().permute(1, 2, 0) + 1) / 2).clamp(0, 1).numpy()
          for v in vistas]

fig, ejes = plt.subplots(2, 5, figsize=(12, 5.2))
ejes[0, 0].imshow(DATOS["train"][0])
ejes[0, 0].set_title("imagen (píxeles)", fontsize=9)
for c in range(4):
    ejes[0, c + 1].imshow(x0[0, c].numpy(), cmap="RdBu_r")
    ejes[0, c + 1].set_title(f"latent, canal {c}", fontsize=9)
ejes[1, 0].imshow(vistas[0])
ejes[1, 0].set_title("x_0 decodificado", fontsize=9)
for j, t in enumerate(TS):
    ejes[1, j + 1].imshow(vistas[j + 1])
    ejes[1, j + 1].set_title(f"x_t con t = {t}, decodificado", fontsize=9)
ejes[1, 4].imshow(xts[-1][0, 0].numpy(), cmap="RdBu_r")
ejes[1, 4].set_title(f"x_t, t = {TS[-1]}, canal 0", fontsize=9)
for ax in ejes.flat:
    ax.axis("off")
fig.suptitle("Arriba: la cara y su latent. Abajo: más t, más ruido", fontsize=11)
fig.tight_layout()
plt.show()

# Ya no hacen falta: liberamos la GPU para entrenar el UNet.
del VAE, TEXTO
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VAE y text encoder liberados.")

## 04 · LoRA y QLoRA en matemáticas

- **Qué vas a ver:** por qué el fine-tuning completo no cabe en una T4, y cómo LoRA y QLoRA mueven muy pocos números.
- Las cifras de esta sección (parámetros, bytes) son cuentas exactas; los números de entrenamiento los imprime el código.

**LoRA (Low-Rank Adaptation).** Cada capa lineal adaptada cambia así:

$$W=W_0+\Delta W,\qquad \Delta W=\frac{\alpha}{r}\,BA,\qquad B\in\mathbb R^{d\times r},\; A\in\mathbb R^{r\times k},\; r\ll\min(d,k).$$

- $W_0\in\mathbb R^{d\times k}$ queda **congelada**; solo se entrenan $A$ y $B$. Escribimos $s=\alpha/r$.
- **Forward sin formar $\Delta W$:** $h=W_0x+s\,B(Ax)$. Primero $Ax\in\mathbb R^{r}$, luego $B(Ax)\in\mathbb R^{d}$.
- **Inicio:** $A$ aleatoria y $B=0$ $\Rightarrow$ $\Delta W=0$: el modelo arranca idéntico al base.
- **Gradientes**, con $g=\partial\mathcal L/\partial h$:

$$\frac{\partial\mathcal L}{\partial B}=s\,g\,(Ax)^\top,\qquad \frac{\partial\mathcal L}{\partial A}=s\,B^\top g\,x^\top.$$

- En el primer backward $B=0$, así que $\partial\mathcal L/\partial A=0$: **solo $B$ se mueve**. (AdamW con weight decay sí mueve $A$ un poco desde el primer paso.)
- **Rango:** $\operatorname{rank}(\Delta W)\le r$, porque $\Delta W$ pasa por un espacio de dimensión $r$.
- **Parámetros:** $dk$ contra $r(d+k)$. Con $d=k=768$ y $r=8$: $589{,}824$ contra $12{,}288$ (≈ 2.1 %).
- **Inferencia:** $W=W_0+\lambda\,s\,BA$ se fusiona en una sola matriz, sin costo extra; $\lambda=0$ recupera el base (§10 mueve $\lambda$).

La celda siguiente construye `LoraLineal` a mano y comprueba el inicio, el primer gradiente, el rango y el conteo.

In [ ]:
# LoRA escrita a mano sobre una lineal 768 → 768 (el ancho de la atención
# de SD 1.5). W0 queda congelada; solo A y B reciben gradiente.
import torch
from torch import nn


class LoraLineal(nn.Module):
    def __init__(self, base, r=8, alfa=8.0):
        super().__init__()
        self.base = base.requires_grad_(False)            # W0 congelada
        d, k = base.out_features, base.in_features
        self.A = nn.Parameter(torch.randn(r, k) / r)      # A aleatoria, r×k
        self.B = nn.Parameter(torch.zeros(d, r))          # B = 0, d×r
        self.s = alfa / r                                 # s = α/r

    def forward(self, x):
        # h = W0 x + s·B(Ax): nunca se forma la matriz d×k de ΔW
        return self.base(x) + self.s * (x @ self.A.T) @ self.B.T

    def delta_w(self):
        return self.s * self.B @ self.A


torch.manual_seed(0)
capa = LoraLineal(nn.Linear(768, 768), r=8, alfa=8.0)
x, y = torch.randn(16, 768), torch.randn(16, 768)

# 1) Al inicio ΔW = 0: la capa da exactamente lo mismo que la base.
print("max |ΔW| al inicio:", capa.delta_w().abs().max().item())
print("max |h − W0 x|:     ", (capa(x) - capa.base(x)).abs().max().item())

# 2) Primer backward: ∂L/∂A = s·Bᵀ g xᵀ = 0 porque B = 0; solo B se mueve.
perdida = ((capa(x) - y) ** 2).mean()
perdida.backward()
print("‖∂L/∂A‖ en el 1.er backward:", capa.A.grad.norm().item())
print("‖∂L/∂B‖ en el 1.er backward:", round(capa.B.grad.norm().item(), 6))

# Tras unos pasos B ≠ 0, A ya recibe gradiente, y rank(ΔW) ≤ r.
opt = torch.optim.SGD([capa.A, capa.B], lr=0.1)
for _ in range(5):
    opt.step()
    opt.zero_grad()
    ((capa(x) - y) ** 2).mean().backward()
print("‖∂L/∂A‖ tras 5 pasos:      ", round(capa.A.grad.norm().item(), 6))
print("rank(ΔW) tras 5 pasos:", torch.linalg.matrix_rank(capa.delta_w()).item(),
      "≤ r = 8")

# 3) Parámetros entrenables: d·k contra r(d + k).
completo = capa.base.weight.numel()
lora = capa.A.numel() + capa.B.numel()
print(f"completo d·k = {completo:,}   LoRA r(d+k) = {lora:,}   "
      f"({100 * lora / completo:.2f} %)")
assert (completo, lora) == (589_824, 12_288)

**Memoria del fine-tuning completo con AdamW (fp32)**

| Qué se guarda por parámetro entrenable | Bytes |
|---|---:|
| Peso | 4 |
| Gradiente | 4 |
| Primer momento $m$ | 4 |
| Segundo momento $v$ | 4 |
| **Total** | **16** |

- UNet de SD 1.5, $P\approx 8.6\times10^8$: $16P\approx 13.8$ GB $=12.8$ GiB, **antes** de activaciones y temporales $\Rightarrow$ no cabe en una T4 de 16 GB.
- Con LoRA, los 16 bytes se pagan solo por los $r(d+k)$ de cada capa adaptada; $W_0$ congelada en fp16 cuesta 2 bytes por parámetro.

In [ ]:
# Memoria de AdamW en fp32: peso 4 + gradiente 4 + m 4 + v 4 = 16 bytes por
# parámetro entrenable, antes de activaciones y temporales. n_unet es el
# UNet real si ya está cargado; si no, el de SD 1.5 (≈ 8.6·10⁸).
# (No se llama P: P es el diccionario de configuración de §1.)
_unet = globals().get("unet")
if isinstance(_unet, nn.Module):
    n_unet, origen = sum(p.numel() for p in _unet.parameters()), "UNet cargado"
else:
    n_unet, origen = 8.6e8, "referencia de SD 1.5"
GB, GIB = 1e9, 2 ** 30
print(f"P = {n_unet:,.0f} parámetros ({origen})")
for que, bytes_por_param in [("completo con AdamW (fp32)", 16),
                             ("solo W0 en fp16", 2),
                             ("solo W0 en NF4 (sin escalas)", 0.5)]:
    b = bytes_por_param * n_unet
    print(f"{que:30s} {b / GB:6.2f} GB = {b / GIB:6.2f} GiB")
print("Colab da ≈ 15 GiB en una T4, y aún faltan las activaciones.")

**QLoRA: LoRA sobre un $W_0$ en 4 bits**

- $W_0$ se guarda en **NF4** por bloques de 64 pesos: $W_0\approx c_b\,q$, con $q$ en 16 niveles (cuantiles de $\mathcal N(0,1)$) y $c_b$ la escala del bloque $b$.
- En cada matmul el bloque se **descuantiza a fp16**; los gradientes solo van a $A$ y $B$, que siguen en fp32.

| | $W_0$ por parámetro | Entrena |
|---|---:|---|
| Completo | 16 bytes (fp32 + AdamW) | todo |
| LoRA | 2 bytes (fp16) | $A,B$ |
| QLoRA | ≈ 0.5 byte (NF4) + escalas | $A,B$ |

- En SD solo se cuantizan las capas **lineales**; las convoluciones quedan en fp16.
- QLoRA **ahorra memoria, no tiempo**: descuantizar cuesta, y en una T4 suele ser más lento que LoRA.
- Con QLoRA, $\lambda=0$ recupera el base **cuantizado**, no el original.

## 05 · La caja de herramientas

Todas las secciones que siguen usan estas funciones **tal como están**. Así, dos corridas solo difieren en lo que cambias en su `Corrida`.

| Función | Qué hace |
|---|---|
| `Corrida(nombre, …)` | Configura un experimento: método, rango $r$, $\alpha$, módulos, lr, pasos, schedule, semilla, datos |
| `entrenar(c)` | Entrena el adaptador, mide y lo guarda; queda en `CORRIDAS[c.nombre]` |
| `perdida_val(unet, latents)` | MSE del ruido con $(t,\varepsilon)$ fijos |
| `generar(nombre, prompts, semillas, escala)` | Imágenes del base (`None`) o con un adaptador |
| `malla`, `curvas` | Ver imágenes y curvas de loss |
| `delta_w`, `norma_relativa` | Abrir el adaptador: $\Delta W$ y su tamaño frente a $W_0$ |
| `embeddings_clip`, `clip_score` | Medir imágenes con CLIP |
| `tabla`, `liberar`, `completo_en_subproceso` | Resumen, memoria, y el fine-tuning completo que no cabe |

Si `entrenar` ya vio una corrida con el mismo nombre y la misma configuración, y no falló, **la reutiliza** sin reentrenar.

In [ ]:
import gc
import json
import math
import tempfile
import time
from dataclasses import dataclass, field

import torch.nn.functional as F
from diffusers import (DDPMScheduler, DPMSolverMultistepScheduler,
                       UNet2DConditionModel)
from diffusers.optimization import get_scheduler
from diffusers.training_utils import cast_training_params
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import load_file, save_file

CUDA = DEVICE.type == "cuda"
CARPETA = "adaptadores"                  # aquí quedan los .safetensors
os.makedirs(CARPETA, exist_ok=True)

In [ ]:
@dataclass
class Corrida:
    nombre: str
    metodo: str = "lora"                 # "lora" o "qlora"
    rango: int = 8
    alfa: float | None = None            # None → alfa = rango
    modulos: tuple = ("to_q", "to_k", "to_v", "to_out.0")
    lr: float = 1e-4
    pasos: int | None = None             # None → P["pasos"] / P["pasos_control"]
    schedule: str = "cosine"             # "cosine" o "constante"
    warmup: int | None = None            # None → 5 % de los pasos
    semilla: int = 0
    datos: str = "train"                 # "train" o "control"
    # --- resultados (los llena entrenar) ---
    perdidas_train: list = field(default_factory=list)   # [(paso, loss)]
    perdidas_val: list = field(default_factory=list)     # [(paso, loss)]
    mem_pico_gb: float = float("nan")
    seg_por_paso: float = float("nan")
    mb_adaptador: float = float("nan")
    n_entrenables: int = 0
    error: str | None = None
    ruta: str | None = None

    def __post_init__(self):
        if self.alfa is None:
            self.alfa = self.rango
        if self.pasos is None:
            self.pasos = P["pasos_control" if self.datos == "control"
                           else "pasos"]
        if self.warmup is None:
            self.warmup = max(1, round(0.05 * self.pasos))

    def config(self):
        """Lo que define la corrida (sin resultados): sirve para no repetirla."""
        return (self.metodo, self.rango, self.alfa, tuple(self.modulos),
                self.lr, self.pasos, self.schedule, self.warmup,
                self.semilla, self.datos)


CORRIDAS: dict[str, Corrida] = {}
_RUIDO = DDPMScheduler.from_pretrained(MODELO, subfolder="scheduler")
print("Corrida por defecto:", Corrida("ejemplo").config())

- **LoRA:** el UNet se carga en fp16 y se congela; solo el adaptador entrena.
- **QLoRA:** las lineales del UNet se cargan en **NF4** (4 bits) con bitsandbytes. La celda cuenta los módulos `Linear4bit` para comprobar que la cuantización ocurrió. Sin GPU, la corrida queda con `error` y el notebook sigue.
- `modulos` es una tupla de nombres: sufijos (`"to_q"`) o nombres completos de capa. Una tupla con **una sola regex** (con `*`, `(` o `|`) se pasa a peft como regex.
- `perdida_val` es la loss de difusión
$$L=\frac{1}{n}\big\lVert \varepsilon-\hat\varepsilon_\theta(x_t,t,c)\big\rVert^2,\qquad x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,$$
con $n$ el número de elementos. Usa siempre los mismos $(t,\varepsilon)$ de `RUIDO_VAL`: la evaluación $i$ toma el latent $i \bmod N$ y el par $i \bmod K$. Por eso dos corridas se miden con la misma vara.

In [ ]:
def _cargar_unet(metodo):
    """UNet congelado: fp16 en GPU (LoRA) o NF4 de 4 bits (QLoRA)."""
    extra = {}
    if metodo == "qlora":
        if not CUDA:
            raise RuntimeError("QLoRA necesita GPU de NVIDIA (bitsandbytes)")
        from diffusers import BitsAndBytesConfig
        extra["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16)
    unet = UNet2DConditionModel.from_pretrained(
        MODELO, subfolder="unet", torch_dtype=DTYPE, **extra)
    if metodo == "qlora":     # ya vive en la GPU; .to() no se permite en 4 bits
        n4 = sum(type(m).__name__ == "Linear4bit" for m in unet.modules())
        print(f"  módulos Linear4bit: {n4}")
        if n4 == 0:
            raise RuntimeError("el UNet no quedó cuantizado (0 Linear4bit)")
    else:
        unet = unet.to(DEVICE)
    unet.requires_grad_(False)
    return unet


def _config_lora(c):
    """LoraConfig de la corrida. Una tupla de nombres va como lista
    (sufijos o nombres completos); un solo texto con '*' o '(' va como regex."""
    m = list(c.modulos)
    if len(m) == 1 and any(ch in m[0] for ch in "*(|"):
        m = m[0]
    return LoraConfig(r=c.rango, lora_alpha=c.alfa, target_modules=m,
                      init_lora_weights="gaussian")


def _latents(datos):
    if datos == "control":
        return LATENTS["train"][:len(CONTROL)]
    return LATENTS[datos]

In [ ]:
@torch.no_grad()
def perdida_val(unet, latents):
    """MSE del ruido con (t, eps) FIJOS de RUIDO_VAL: la evaluación i usa
    el latent i mod N y el par i mod K, así que dos corridas se miden igual."""
    t_fijo, eps_fijo = RUIDO_VAL
    n, k = len(latents), len(t_fijo)
    idx = torch.arange(max(n, k))
    dt = next(unet.parameters()).dtype if not CUDA else torch.float16
    emb = EMB.to(DEVICE, dt)
    estaba = unet.training
    unet.eval()
    total, cuenta = 0.0, 0
    for lote in idx.split(2 * P["batch"]):
        x0 = latents[lote % n].to(DEVICE, dt)
        eps = eps_fijo[lote % k].to(DEVICE, dt)
        t = t_fijo[lote % k].to(DEVICE)
        xt = _RUIDO.add_noise(x0, eps, t)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
            pred = unet(xt, t, encoder_hidden_states=emb.expand(len(x0), -1, -1)
                        ).sample
        total += F.mse_loss(pred.float(), eps.float(), reduction="sum").item()
        cuenta += eps.numel()
    unet.train(estaba)
    return total / cuenta

Un paso de entrenamiento:

1. Toma un batch de latents $x_0$, sortea $\varepsilon\sim\mathcal N(0,I)$ y $t$ uniforme, y arma $x_t$.
2. El UNet predice $\hat\varepsilon$; la loss es la MSE.
3. `backward` llega **solo** a $A$ y $B$; AdamW (weight decay $10^{-2}$) los actualiza. El gradiente se recorta a norma 1.

Detalles que importan en la T4:

| Pieza | Por qué |
|---|---|
| UNet fp16, adaptador fp32 (`cast_training_params`) | `GradScaler` no sabe desescalar gradientes fp16 |
| `autocast` + `GradScaler` | fp16 rápido sin que los gradientes chicos se vuelvan cero |
| warmup + cosine (o constante) | el lr sube los primeros pasos (5 % por defecto) y luego baja |
| gradient checkpointing | guarda menos activaciones y las recalcula en el backward: menos memoria, más tiempo |
| paso con inf/NaN | `GradScaler` lo salta y baja la escala; ese intento no cuenta como paso |
| `mem_pico_gb` | memoria pico de la GPU durante la corrida (antes se suelta el pipeline; GB = $10^9$ bytes) |
| `seg_por_paso` | promedio de los pasos 5–15, tras sincronizar la GPU |

Al final guarda solo $A$ y $B$ en `adaptadores/<nombre>.safetensors` y reporta sus MB.

In [ ]:
def _preparar(c):
    """UNet congelado + adaptador fp32 + AdamW + schedule + GradScaler."""
    liberar()                        # suelta el pipeline y CLIP: GPU para entrenar
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    unet = _cargar_unet(c.metodo)
    unet.add_adapter(_config_lora(c))
    unet.enable_gradient_checkpointing()   # menos activaciones, más cómputo
    # Adaptador en fp32 sobre el UNet fp16: GradScaler no desescala fp16.
    cast_training_params([unet], dtype=torch.float32)
    entrenables = [p for p in unet.parameters() if p.requires_grad]
    c.n_entrenables = sum(p.numel() for p in entrenables)
    opt = torch.optim.AdamW(entrenables, lr=c.lr, weight_decay=1e-2)
    sched = get_scheduler("cosine" if c.schedule == "cosine"
                          else "constant_with_warmup", opt,
                          num_warmup_steps=c.warmup, num_training_steps=c.pasos)
    return unet, entrenables, opt, sched, torch.amp.GradScaler("cuda", enabled=CUDA)

In [ ]:
def _entrenar(c):
    semilla(c.semilla)
    unet, entrenables, opt, sched, escalador = _preparar(c)
    X, g = _latents(c.datos), torch.Generator().manual_seed(c.semilla)
    emb = EMB.to(DEVICE, DTYPE)
    tiempos, paso, intentos = [], 0, 0
    unet.train()
    while paso < c.pasos and intentos < 2 * c.pasos:
        intentos += 1
        t0 = time.perf_counter()
        x0 = X[torch.randint(len(X), (P["batch"],), generator=g)]
        eps = torch.randn(x0.shape, generator=g)
        t = torch.randint(0, _RUIDO.config.num_train_timesteps,
                          (len(x0),), generator=g)
        x0, eps, t = x0.to(DEVICE, DTYPE), eps.to(DEVICE, DTYPE), t.to(DEVICE)
        xt = _RUIDO.add_noise(x0, eps, t)              # forward de difusión
        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
            pred = unet(xt, t, encoder_hidden_states=emb.expand(len(x0), -1, -1)
                        ).sample
        loss = F.mse_loss(pred.float(), eps.float())   # ‖ε − ε̂‖² promedio
        opt.zero_grad(set_to_none=True)
        escalador.scale(loss).backward()
        escalador.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(entrenables, 1.0)
        escala_previa = escalador.get_scale()
        escalador.step(opt)
        escalador.update()
        if escalador.get_scale() < escala_previa:      # inf/NaN: paso saltado
            continue
        paso += 1
        sched.step()
        if CUDA:
            torch.cuda.synchronize()
        tiempos.append(time.perf_counter() - t0)
        c.perdidas_train.append((paso, loss.item()))
        if paso % P["val_cada"] == 0 or paso == c.pasos:
            c.perdidas_val.append((paso, perdida_val(unet, LATENTS["validation"])))
    _cerrar(c, unet, tiempos)


def _cerrar(c, unet, tiempos):
    """Memoria pico, s/paso y el adaptador (solo A y B) a disco."""
    if CUDA:
        c.mem_pico_gb = torch.cuda.max_memory_allocated() / 1e9
    medidos = tiempos[4:15] if len(tiempos) >= 15 else tiempos[1:] or tiempos
    c.seg_por_paso = sum(medidos) / max(len(medidos), 1)
    estado = {k: v.detach().float().cpu().contiguous()
              for k, v in get_peft_model_state_dict(unet).items()}
    c.ruta = os.path.join(CARPETA, f"{c.nombre}.safetensors")
    save_file(estado, c.ruta)
    c.mb_adaptador = os.path.getsize(c.ruta) / 1e6

In [ ]:
def entrenar(c: Corrida) -> Corrida:
    """Entrena (o reutiliza) la corrida y la registra en CORRIDAS.
    Un error (OOM, bitsandbytes…) queda en c.error y el notebook sigue."""
    previa = CORRIDAS.get(c.nombre)
    if (previa is not None and previa.error is None
            and previa.config() == c.config()):
        print(f"{c.nombre}: ya estaba entrenada, la reutilizo")
        return previa
    _quitar_del_pipe(c.nombre)       # si se reentrena, el pipe la recarga
    print(f"{c.nombre}: {c.metodo} r={c.rango} α={c.alfa} lr={c.lr} "
          f"pasos={c.pasos} ({c.schedule}) datos={c.datos}")
    inicio = time.perf_counter()
    try:
        _entrenar(c)
    except Exception as e:           # OOM, bitsandbytes, sin GPU…
        c.error = f"{type(e).__name__}: {str(e).splitlines()[0][:200]}"
        print(f"  ✗ {c.error}")
    finally:
        gc.collect()
        if CUDA:
            torch.cuda.empty_cache()
    CORRIDAS[c.nombre] = c
    if c.error is None:
        print(f"  ✓ {time.perf_counter() - inicio:.0f} s · val final "
              f"{c.perdidas_val[-1][1]:.4f} · {c.mb_adaptador:.1f} MB")
    return c

- `generar` usa **un solo** pipeline base (fp16, `DPMSolverMultistepScheduler`, 20 pasos), en caché, y genera de 2 en 2 semillas. `entrenar` lo suelta; la siguiente llamada lo vuelve a cargar.
- Con un adaptador, lo carga encima del UNet base y lo activa; con `None` los apaga todos.
- `escala` es $\lambda$ en $W = W_0 + \lambda\,\frac{\alpha}{r}BA$: con $\lambda=0$ sale el base, con $\lambda=1$ el adaptado.
- **Un adaptador QLoRA se aplica sobre el UNet fp16**, no sobre el cuantizado en que se entrenó. La diferencia NF4 → fp16 es pequeña, pero existe.
- Misma semilla ⇒ mismo ruido inicial: la diferencia entre dos imágenes con igual semilla es efecto del adaptador.

In [ ]:
_PIPE, _EN_PIPE = None, set()        # pipeline base en caché y adaptadores cargados


def _pipe():
    """StableDiffusionPipeline base (fp16 en GPU) armado pieza por pieza."""
    global _PIPE
    if _PIPE is None:
        from diffusers import AutoencoderKL, StableDiffusionPipeline
        from transformers import CLIPTextModel, CLIPTokenizer
        _PIPE = StableDiffusionPipeline(
            vae=AutoencoderKL.from_pretrained(MODELO, subfolder="vae",
                                              torch_dtype=DTYPE),
            text_encoder=CLIPTextModel.from_pretrained(
                MODELO, subfolder="text_encoder", torch_dtype=DTYPE),
            tokenizer=CLIPTokenizer.from_pretrained(MODELO, subfolder="tokenizer"),
            unet=UNet2DConditionModel.from_pretrained(
                MODELO, subfolder="unet", torch_dtype=DTYPE),
            scheduler=DPMSolverMultistepScheduler.from_pretrained(
                MODELO, subfolder="scheduler"),
            safety_checker=None, feature_extractor=None,
            requires_safety_checker=False).to(DEVICE)
        _PIPE.set_progress_bar_config(disable=True)
        _PIPE.vae.enable_slicing()     # decodifica de una en una: menos VRAM
        _EN_PIPE.clear()
    return _PIPE

In [ ]:
def _quitar_del_pipe(nombre):
    if _PIPE is not None and nombre in _adaptadores(_PIPE.unet):
        _PIPE.unet.delete_adapters(nombre)
    _EN_PIPE.discard(nombre)


def _adaptadores(unet):
    """Nombres de adaptadores que el UNet tiene de verdad (sin peft: ninguno)."""
    return set(getattr(unet, "peft_config", None) or {})


def _activar(nombre):
    """Pone el adaptador `nombre` (o ninguno, si es None) en el UNet del pipe.
    Un adaptador QLoRA se aplica sobre el UNet fp16, no sobre el cuantizado."""
    unet = _pipe().unet
    _EN_PIPE.intersection_update(_adaptadores(unet))   # sincroniza con el UNet
    if nombre is None:
        if _EN_PIPE:
            unet.disable_adapters()
        return
    c = CORRIDAS[nombre]
    if c.error is not None:
        raise ValueError(f"la corrida {nombre} falló: {c.error}")
    if nombre not in _EN_PIPE:
        unet.add_adapter(_config_lora(c), adapter_name=nombre)
        set_peft_model_state_dict(unet, load_file(c.ruta), adapter_name=nombre)
        _EN_PIPE.add(nombre)
    unet.set_adapter(nombre)
    unet.enable_adapters()


@torch.no_grad()
def generar(nombre, prompts=PROMPT, semillas=None, escala=1.0):
    """Una imagen por (prompt, semilla), prompt-mayor. nombre=None → base.
    escala = λ multiplica a ΔW (λ=0 es el base)."""
    prompts = [prompts] if isinstance(prompts, str) else list(prompts)
    semillas = P["semillas"] if semillas is None else list(semillas)
    _activar(nombre)
    pipe, salida = _pipe(), []
    # sin adaptador no se pasa scale: diffusers nuevo exige peft_config
    extra = {} if nombre is None else {"cross_attention_kwargs": {"scale": escala}}
    for p in prompts:                # lotes de 2 semillas: cabe en la T4
        for i in range(0, len(semillas), 2):
            lote = semillas[i:i + 2]
            gens = [torch.Generator("cpu").manual_seed(s) for s in lote]
            salida += pipe([p] * len(lote), generator=gens,
                           num_inference_steps=2 if RAPIDO else 20,
                           height=P["res"], width=P["res"],
                           **extra).images
    return salida

- `malla(imagenes, filas, titulo, etiquetas)`: las imágenes en filas (reducidas a 256 px). `etiquetas` puede traer una por imagen o una por fila.
- `curvas(nombres, titulo)`: loss contra paso. **Train** es delgada y tenue (cada paso usa otro $t$, por eso brinca); **validation** es gruesa con puntos.

**Cómo leer las curvas:** compara las gruesas entre corridas; la delgada solo da la tendencia.

In [ ]:
def malla(imagenes, filas, titulo, etiquetas=None):
    """Malla de imágenes (≤ 256 px, dpi 80). etiquetas: una por imagen
    (título chico) o una por fila (a la izquierda)."""
    n = len(imagenes)
    cols = max(1, math.ceil(n / filas))
    fig, ejes = plt.subplots(filas, cols, figsize=(1.9 * cols, 1.9 * filas),
                             squeeze=False, dpi=80)
    for i, eje in enumerate(ejes.flat):
        eje.axis("off")
        if i >= n:
            continue
        im = imagenes[i].copy()
        im.thumbnail((256, 256))
        eje.imshow(im)
        if etiquetas is not None and len(etiquetas) == n:
            eje.set_title(etiquetas[i], fontsize=7)
    if etiquetas is not None and len(etiquetas) == filas and filas != n:
        for f in range(filas):
            ejes[f, 0].text(-0.05, 0.5, etiquetas[f], fontsize=8, ha="right",
                            va="center", transform=ejes[f, 0].transAxes)
    fig.suptitle(titulo, fontsize=10)
    plt.tight_layout()
    plt.show()


def curvas(nombres, titulo):
    """Loss por paso: train delgada y tenue, validation gruesa; un color
    por corrida. Las corridas con error se avisan y no se dibujan."""
    fig, eje = plt.subplots(figsize=(7.5, 3.6))
    for i, n in enumerate(nombres):
        c = CORRIDAS.get(n)
        if c is None or c.error is not None:
            print(f"(sin curva para {n}: {c.error if c else 'no existe'})")
            continue
        color = plt.cm.tab10(i % 10)
        if c.perdidas_train:
            eje.plot(*zip(*c.perdidas_train), color=color, lw=0.8, alpha=0.35,
                     label=f"{n} · train")
        if c.perdidas_val:
            eje.plot(*zip(*c.perdidas_val), color=color, lw=2.5, marker="o",
                     ms=3, label=f"{n} · validation")
    eje.set(xlabel="paso", ylabel="MSE del ruido", title=titulo)
    eje.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()

- `delta_w(nombre)`: por cada capa adaptada, $\Delta W=\frac{\alpha}{r}BA$, de forma $[d_{\text{out}}, d_{\text{in}}]$ y rango $\le r$.
- `norma_relativa(nombre)`: $\lVert\Delta W\rVert_F/\lVert W_0\rVert_F$ por capa. Dice **cuánto** se movió cada capa respecto a su tamaño.
- `embeddings_clip(imagenes)`: vectores de CLIP con norma 1; el coseno entre dos imágenes es su producto punto.
- `clip_score(imagenes, prompt)`: $100\cdot\cos(\text{imagen},\text{texto})$ promediado. Más alto = la imagen se parece más al texto según CLIP.

In [ ]:
def delta_w(nombre):
    """ΔW = (α/r)·B·A por capa adaptada, en CPU y fp32."""
    c = CORRIDAS[nombre]
    estado, s = load_file(c.ruta), c.alfa / c.rango
    salida = {}
    for k in estado:
        if k.endswith(".lora_A.weight"):
            capa = k[: -len(".lora_A.weight")]
            B = estado[capa + ".lora_B.weight"].float()
            salida[capa] = s * (B @ estado[k].float())     # [d_out, d_in]
    return salida


_NORMA_W0 = {}


def norma_relativa(nombre):
    """‖ΔW‖_F / ‖W₀‖_F por capa (índice = nombre de la capa)."""
    if not _NORMA_W0:
        for n, m in _pipe().unet.named_modules():
            if isinstance(m, torch.nn.Linear):
                n = n.removesuffix(".base_layer")
                _NORMA_W0[n] = m.weight.float().norm().item()
    dw = delta_w(nombre)
    return pd.Series({k: v.norm().item() / _NORMA_W0[k] for k, v in dw.items()},
                     name=nombre)

In [ ]:
_CLIP = None


def _clip():
    global _CLIP
    if _CLIP is None:
        from transformers import CLIPModel, CLIPProcessor
        _CLIP = (CLIPModel.from_pretrained(MODELO_CLIP).to(DEVICE).eval(),
                 CLIPProcessor.from_pretrained(MODELO_CLIP))
    return _CLIP


def _vector(salida):
    """get_*_features devuelve un tensor o (versiones nuevas) un objeto."""
    if torch.is_tensor(salida):
        return salida
    for campo in ("image_embeds", "text_embeds", "pooler_output"):
        if getattr(salida, campo, None) is not None:
            return getattr(salida, campo)
    raise TypeError(type(salida))


@torch.no_grad()
def embeddings_clip(imagenes):
    """(N, D) con norma 1: el coseno entre dos imágenes es su producto punto."""
    modelo, proc = _clip()
    out = []
    for i in range(0, len(imagenes), 32):
        x = proc(images=imagenes[i:i + 32], return_tensors="pt").to(DEVICE)
        out.append(_vector(modelo.get_image_features(**x)).float().cpu())
    return F.normalize(torch.cat(out), dim=-1)


@torch.no_grad()
def clip_score(imagenes, prompt):
    """CLIP score = 100 · coseno(imagen, texto), promediado (más es mejor)."""
    modelo, proc = _clip()
    x = proc(text=[prompt], return_tensors="pt", padding=True).to(DEVICE)
    texto = F.normalize(_vector(modelo.get_text_features(**x)).float().cpu(), dim=-1)
    return float(100 * (embeddings_clip(imagenes) @ texto.T).mean())

- `tabla()` junta todas las corridas: lo configurado (método, rango, módulos, lr, pasos) y lo medido (loss de validation final, memoria, s/paso, MB, parámetros entrenables, error).
- `liberar()` suelta el pipeline y CLIP de la memoria; la siguiente llamada a `generar` los vuelve a cargar.

In [ ]:
def tabla(nombres=None):
    """Una fila por corrida: lo que se configuró y lo que se midió."""
    filas = []
    for n in (nombres or list(CORRIDAS)):
        c = CORRIDAS[n]
        filas.append({
            "corrida": n, "método": c.metodo, "rango": c.rango,
            "módulos": ",".join(c.modulos) if len(c.modulos) <= 6
            else f"{len(c.modulos)} capas",
            "lr": c.lr, "pasos": c.pasos,
            "loss val final": c.perdidas_val[-1][1] if c.perdidas_val
            else float("nan"),
            "mem pico GB": c.mem_pico_gb, "s/paso": c.seg_por_paso,
            "MB adaptador": c.mb_adaptador,
            "parámetros entrenables": c.n_entrenables, "error": c.error})
    return pd.DataFrame(filas).set_index("corrida")


def liberar():
    """Suelta el pipeline y CLIP de la memoria (GPU y RAM)."""
    global _PIPE, _CLIP
    _PIPE, _CLIP = None, None
    _EN_PIPE.clear()
    _NORMA_W0.clear()
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()

- El fine-tuning **completo** entrena los ~$8.6\cdot10^8$ pesos del UNet en fp32 con AdamW: pesos + gradientes + dos momentos ≈ 16 bytes por peso, antes de activaciones.
- Se intenta en **otro proceso**: un paso con batch 1, hasta el primer `optimizer.step()`. Si se queda sin memoria (OOM), el proceso muere y la GPU del notebook queda intacta.
- Devuelve `{"ok", "error", "mem_pico_gb"}`. Sin GPU devuelve `error="sin GPU"`.

In [ ]:
_GUION_COMPLETO = r'''
import json, sys, torch, torch.nn.functional as F
from diffusers import UNet2DConditionModel
modelo, lado = sys.argv[1], int(sys.argv[2])
r = {"ok": False, "error": None, "mem_pico_gb": None}
try:
    unet = UNet2DConditionModel.from_pretrained(
        modelo, subfolder="unet", torch_dtype=torch.float32).to("cuda").train()
    opt = torch.optim.AdamW(unet.parameters(), lr=1e-5,   # TODO el UNet
                            foreach=False)
    x = torch.randn(1, 4, lado, lado, device="cuda")      # batch 1
    t = torch.randint(0, 1000, (1,), device="cuda")
    emb = torch.randn(1, 77, unet.config.cross_attention_dim, device="cuda")
    pred = unet(x, t, encoder_hidden_states=emb).sample
    F.mse_loss(pred, torch.randn_like(pred)).backward()
    opt.step()                       # aquí nacen los dos momentos de Adam
    torch.cuda.synchronize()
    r["ok"] = True
except torch.cuda.OutOfMemoryError as e:
    r["error"] = "OOM: " + str(e).splitlines()[0][:160]
except Exception as e:
    r["error"] = f"{type(e).__name__}: {str(e).splitlines()[0][:160]}"
if torch.cuda.is_available():
    r["mem_pico_gb"] = torch.cuda.max_memory_allocated() / 1e9
print("JSON" + json.dumps(r))
'''


def completo_en_subproceso():
    """Un paso de fine-tuning COMPLETO (fp32 + AdamW) en otro proceso:
    si revienta por memoria, la VRAM se libera al terminar el proceso."""
    if not CUDA:
        return {"ok": False, "error": "sin GPU", "mem_pico_gb": None}
    liberar()                        # que el otro proceso tenga la GPU entera
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(_GUION_COMPLETO)
    lado = LATENTS["train"].shape[-1]
    try:
        r = subprocess.run([sys.executable, f.name, MODELO, str(lado)],
                           capture_output=True, text=True, timeout=1200)
        linea = [l for l in r.stdout.splitlines() if l.startswith("JSON")]
        if not linea:
            cola = (r.stderr or "").strip().splitlines()[-1:] or ["sin salida"]
            return {"ok": False, "error": cola[0][:200], "mem_pico_gb": None}
        return json.loads(linea[-1][4:])
    except Exception as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}"[:200],
                "mem_pico_gb": None}

## 06 · Antes de entrenar

- El modelo base genera caras con «an anime face». Cada semilla de `P["semillas"]` fija el ruido inicial $x_T$.
- **Cómo leer la malla.** Una columna por semilla. Mira estilo, proporciones y fondo. En s07 volvemos a generar esta referencia con el mismo prompt y las mismas semillas.

In [ ]:
# Modelo base (nombre=None), una imagen por semilla. Las mismas semillas
# vuelven en s07 para que cada columna compare el mismo ruido inicial.
IMGS_BASE_S06 = generar(None, PROMPT, P["semillas"])
malla(IMGS_BASE_S06, filas=1,
      titulo=f"Modelo base · «{PROMPT}»",
      etiquetas=[f"semilla {s}" for s in P["semillas"]])

El modelo base ya puede producir rasgos de anime. LoRA ajusta esa capacidad al estilo del dataset.

| Fila | Prompt | Qué mirar |
|---|---|---|
| 1 | «a face» | Qué estilo elige el modelo por defecto |
| 2 | «an anime face, colorful illustration, detailed» | Cómo cambia al pedir una ilustración detallada |

**Cómo leer la malla.** Compara las dos filas dentro de cada columna. Se usan las dos primeras semillas de la malla anterior.

In [ ]:
# Dos prompts vecinos: uno sin «anime» y uno que pide el estilo con más
# detalle. Mismas semillas: solo cambia el texto.
PROMPTS_S06 = ["a face", "an anime face, colorful illustration, detailed"]
SEMILLAS_S06 = P["semillas"][:2]
imgs = generar(None, PROMPTS_S06, SEMILLAS_S06)       # orden prompt-mayor
malla(imgs, filas=len(PROMPTS_S06),
      titulo="Modelo base con dos prompts de comparación",
      etiquetas=[f"«{p}»" for p in PROMPTS_S06])     # una por fila

## 07 · Los cuatro casos

| Caso | Qué se actualiza | Precisión de los pesos base del UNet |
|---|---|---|
| Completo | Todo el UNet (solo se intenta un paso) | fp32 |
| LoRA $r=8$ | $A$ y $B$ de las proyecciones de atención | fp16 en GPU |
| QLoRA $r=8$ | $A$ y $B$ de las mismas proyecciones | NF4 en las capas lineales |
| ➕ LoRA $r=1$ y $r=64$ | $A$ y $B$ con otro rango | fp16 en GPU |

Con AdamW en fp32, cada parámetro entrenable requiere cuatro valores de 4 bytes.

$$
\underbrace{4}_{W}+\underbrace{4}_{\nabla W}
+\underbrace{4}_{m}+\underbrace{4}_{v}=16\ \text{bytes}.
$$

Para el UNet de Stable Diffusion 1.5, la cuenta da aproximadamente 13.8 GB (12.8 GiB) antes de activaciones y temporales. En una T4 (~15 GiB) esperamos OOM. El subproceso registra si logró dar un paso o qué error encontró.

- **Raro posible:** en una GPU grande (L4, A100) el completo puede caber. No contradice la cuenta: 12.8 GiB más activaciones caben en 22 o 40 GiB.
- **Apuesta.** ¿QLoRA dará caras peores que LoRA?

In [ ]:
import pandas as pd
import torch
from diffusers import UNet2DConditionModel

# Contamos los parámetros del UNet sin cargar pesos: en el device «meta»
# los tensores no ocupan memoria (solo se baja el config.json).
try:
    config_s07 = UNet2DConditionModel.load_config(MODELO, subfolder="unet")
    with torch.device("meta"):
        unet_vacio = UNet2DConditionModel.from_config(config_s07)
    N_UNET_S07 = sum(p.numel() for p in unet_vacio.parameters())
    origen = "contado del config"
    del unet_vacio
except Exception as e:                       # sin red, versión rara, etc.
    N_UNET_S07, origen = int(8.6e8), f"aproximado ({type(e).__name__})"

# AdamW en fp32: cada parámetro ENTRENABLE paga cuatro copias de 4 bytes.
PARTES_S07 = {"pesos (fp32)": 4, "gradiente": 4, "m de AdamW": 4,
              "v de AdamW": 4}
filas = [{"qué": k, "bytes/param": b, "GB": N_UNET_S07 * b / 1e9,
          "GiB": N_UNET_S07 * b / 2**30} for k, b in PARTES_S07.items()]
filas.append({"qué": "TOTAL", "bytes/param": 16, "GB": N_UNET_S07 * 16 / 1e9,
              "GiB": N_UNET_S07 * 16 / 2**30})
print(f"Parámetros del UNet: {N_UNET_S07:,} ({origen})")
display(pd.DataFrame(filas).set_index("qué").round(2))
print("…y eso es ANTES de activaciones, temporales y el resto del pipeline.")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"Memoria de esta GPU: {total:.1f} GiB")

In [ ]:
# El completo corre en OTRO proceso: si se queda sin memoria, este notebook
# sigue vivo, y al terminar ese proceso la GPU queda libre otra vez.
liberar()            # soltamos el pipeline de s06 para no competir por VRAM
RES_COMPLETO_S07 = completo_en_subproceso()
if RES_COMPLETO_S07["ok"]:
    print("El fine-tuning completo CUPO: dio un paso de AdamW con "
          f"{RES_COMPLETO_S07['mem_pico_gb']:.2f} GB de pico. "
          "Resultado raro en una T4: lee el comentario de abajo.")
else:
    print(f"El fine-tuning completo NO corrió: {RES_COMPLETO_S07['error']}")
    if RES_COMPLETO_S07.get("mem_pico_gb"):
        print(f"memoria pico antes de fallar: "
              f"{RES_COMPLETO_S07['mem_pico_gb']:.2f} GB")

- LoRA $r=8$ sirve de referencia. Si termina, se muestran los segundos por paso y el tiempo estimado para este modo.
- QLoRA puede ahorrar memoria al guardar los pesos lineales en NF4. Descuantizarlos durante el cálculo puede alargar cada paso. La comparación de tiempos aparece si ambas corridas terminan.
- ➕ Al variar $r$ cambia la capacidad del adaptador y se mantiene $\alpha/r=1$. **Apuesta.** ¿Se verá alguna diferencia entre $r=1$, $r=8$ y $r=64$?

In [ ]:
# LoRA r = 8 en la atención (to_q, to_k, to_v, to_out.0), lr 1e-4, cosine.
# Es la corrida base del notebook: la usan s08, s10, s11, s12 y s13.
R8_S07 = entrenar(Corrida("lora_r8"))
if R8_S07.error:
    print(f"lora_r8 falló: {R8_S07.error}")
elif R8_S07.seg_por_paso:
    # Estimación de tiempo con el s/paso medido (tras calentar la GPU).
    minutos = R8_S07.seg_por_paso * P["pasos"] / 60
    print(f"lora_r8: {R8_S07.seg_por_paso:.3f} s/paso → una corrida LoRA de "
          f"este MODO ({P['pasos']} pasos) tarda ≈ {minutos:.1f} min")

# QLoRA: el mismo adaptador, pero W₀ guardada en 4 bits NF4.
# Sin CUDA o sin bitsandbytes la caja deja el error en la corrida y sigue.
Q8_S07 = entrenar(Corrida("qlora_r8", metodo="qlora"))
if Q8_S07.error:
    print(f"qlora_r8 no corrió: {Q8_S07.error} (el notebook sigue)")
elif not R8_S07.error and R8_S07.seg_por_paso:
    razon = Q8_S07.seg_por_paso / R8_S07.seg_por_paso
    print(f"qlora_r8: {Q8_S07.seg_por_paso:.3f} s/paso = {razon:.2f}× LoRA")

In [ ]:
# ➕ El rango: r = 1 (casi nada) y r = 64 (8× más parámetros que r = 8).
# alfa = rango en las tres, así la escala α/r vale 1 en todas.
if LARGO:
    for r in (1, 64):
        c = entrenar(Corrida(f"lora_r{r}", rango=r))
        print(f"lora_r{r}: " + (c.error or "lista"))
else:
    print("(solo en MODO largo)")

- **Cómo leer la tabla.** Una fila por corrida registrada. Compara memoria pico, segundos por paso y tamaño del adaptador guardado. La columna de error explica las corridas fallidas.
- Debajo aparece la fracción de parámetros entrenables respecto al UNet y su costo estimado con 16 bytes por parámetro. En modo `RAPIDO` se usa un modelo diminuto para probar el flujo.

In [ ]:
NOMBRES_S07 = [n for n in ("lora_r8", "qlora_r8", "lora_r1", "lora_r64")
               if n in CORRIDAS]
OK_S07 = [n for n in NOMBRES_S07 if CORRIDAS[n].error is None]
display(tabla(NOMBRES_S07))

# Lo entrenable frente al completo, y lo que AdamW paga por ello (16 B/param).
for n in OK_S07:
    c = CORRIDAS[n]
    print(f"{n:>9}: {c.n_entrenables:>10,} entrenables = "
          f"{100 * c.n_entrenables / N_UNET_S07:.2f} % del UNet · "
          f"AdamW: {16 * c.n_entrenables / 2**20:.1f} MiB "
          f"(vs {16 * N_UNET_S07 / 2**30:.1f} GiB el completo)")
if RAPIDO:
    print("Modo RAPIDO: modelo diminuto y unos pocos pasos; la tabla solo "
          "prueba que el código corre.")

- Si $\varepsilon_j$ es el ruido real, $\varepsilon_{\theta,j}(x_t,t,c)$ es la predicción y $D$ es el número de elementos, la loss es la MSE, $\frac{1}{D}\sum_{j=1}^{D}(\varepsilon_j-\varepsilon_{\theta,j}(x_t,t,c))^2$.
- **Cómo leer las curvas.** Un color por corrida exitosa, línea delgada para train y gruesa para validation. En train se sortean $t$ y $\varepsilon$. En validation se reutilizan para facilitar la comparación.

In [ ]:
if OK_S07:
    curvas(OK_S07, "Los casos que corrieron: train (delgada) y validation (gruesa)")
else:
    print("Ninguna corrida terminó: no hay curvas que mostrar.")

- **Cómo leer la malla.** El prompt es fijo. Arriba está el modelo base y debajo las corridas exitosas. Cada columna comparte la semilla con s06.
- La fila de QLoRA se genera con el adaptador sobre el UNet **fp16**, no sobre el cuantizado (así lo hace la caja, s05).
- Con pocos pasos puede haber cambios pequeños. El adaptador arranca con $\Delta W=0$.
- Debajo se imprime la diferencia absoluta media de píxeles respecto al base, en una escala de 0 a 255. Úsala para medir cuánto cambió la imagen y mira la malla para describir el cambio.

In [ ]:
import numpy as np

# Una fila por método, las MISMAS semillas de s06 en cada columna.
FILAS_S07 = [None] + OK_S07                     # None = modelo base
imgs = []
for nombre in FILAS_S07:
    imgs += generar(nombre, PROMPT, P["semillas"])
n_sem = len(P["semillas"])
malla(imgs, filas=len(FILAS_S07),
      titulo=f"«{PROMPT}» · una fila por método, mismas semillas",
      etiquetas=[n or "base" for n in FILAS_S07])    # una por fila

# ¿Cuánto se movió cada fila respecto al base? Diferencia media de píxeles
# (0 = idéntica, 255 = opuesta), semilla por semilla.
base = [np.asarray(im, dtype=np.float32) for im in imgs[:n_sem]]
for i, nombre in enumerate(OK_S07, start=1):
    fila = imgs[i * n_sem:(i + 1) * n_sem]
    dif = [np.abs(np.asarray(im, dtype=np.float32) - b).mean()
           for im, b in zip(fila, base)]
    print(f"{nombre:>9} vs base: |Δpíxel| medio = {np.mean(dif):.1f} / 255")

**Qué mirar**

| Qué | Qué esperar | Qué puede salir raro, y por qué |
|---|---|---|
| Completo | OOM en una T4 | Cabe en una GPU grande; la cuenta de 16 B/param sigue siendo cierta |
| Memoria pico | QLoRA < LoRA | Ahorra menos de lo que sugieren los 4 bits: las convoluciones y las activaciones no se cuantizan |
| Segundos por paso | QLoRA más lento | Descuantizar $W_0$ en cada matmul cuesta: ahorra memoria, no tiempo |
| Calidad (loss y malla) | QLoRA ≈ LoRA | Las curvas de validation pueden quedar cerca aunque cambien estilo, rasgos o fondo; la malla es el mejor juez |
| Rango ➕ | poca diferencia | $r=1$ ya mueve el estilo; $r=64$ tiene más capacidad, pero con pocos pasos puede no usarla |
| Cambio vs base | caras más anime | Con pocos pasos las imágenes pueden cambiar poco: el adaptador arranca en $\Delta W=0$ |

- Si una corrida dijo «no corrió» (sin GPU, sin `bitsandbytes`), falta en la tabla y en la malla. El resto del notebook usa `lora_r8`.

## 08 · Dónde poner LoRA

- LoRA no va en todo el UNet: va en las capas lineales que nombra `target_modules`. Aquí, por omisión, las cuatro proyecciones de atención `to_q, to_k, to_v, to_out.0`.
- El UNet tiene tres zonas: **down** (baja la resolución), **mid** (la más chica) y **up** (la sube y reconstruye el detalle).
- Cada zona tiene **self-attention** (`attn1`: la imagen consigo misma) y **cross-attention** (`attn2`: la imagen consulta el texto del prompt).
- Medimos cuánto se movió cada capa relativo a su tamaño:

$$\frac{\lVert\Delta W\rVert_F}{\lVert W_0\rVert_F},\qquad \lVert M\rVert_F=\sqrt{\textstyle\sum_{ij}M_{ij}^2}$$

**Cómo leer la gráfica.** Izquierda: una barra por capa, en orden down → mid → up, color por zona. Derecha: el promedio por zona y tipo de atención. Es una sola corrida y no hay barras de error: diferencias chicas entre zonas pueden ser ruido.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ZONAS_S08 = {"down": "down_blocks", "mid": "mid_block", "up": "up_blocks"}
COLOR_S08 = {"down": "#9467bd", "mid": "#8c564b", "up": "#17becf"}


def zona_s08(capa):
    """down / mid / up según el nombre de la capa en el UNet."""
    return next((z for z, n in ZONAS_S08.items() if n in capa), "otra")


def tipo_s08(capa):
    """attn1 = self-attention, attn2 = cross-attention, ff = MLP."""
    if "attn1" in capa:
        return "self-attention"
    if "attn2" in capa:
        return "cross-attention"
    return "MLP" if ".ff." in capa else "otra"


def por_capa_s08(nombre):
    """norma_relativa de una corrida, en orden down → mid → up."""
    s = norma_relativa(nombre)
    df = pd.DataFrame({"capa": s.index, "norma": s.values})
    df["zona"] = df["capa"].map(zona_s08)
    df["tipo"] = df["capa"].map(tipo_s08)
    orden = {z: i for i, z in enumerate(list(ZONAS_S08) + ["otra"])}
    # sort estable: dentro de cada zona se conserva el orden del UNet
    return df.sort_values("zona", key=lambda z: z.map(orden), kind="stable")


R8_S08 = CORRIDAS.get("lora_r8")
if R8_S08 is None or R8_S08.error:
    print("lora_r8 no está lista (sección 07): no hay ΔW que medir.")
else:
    df = por_capa_s08("lora_r8").reset_index(drop=True)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4),
                                 gridspec_kw={"width_ratios": [2, 1]})
    a1.bar(df.index, df["norma"], color=df["zona"].map(COLOR_S08).fillna("gray"))
    for z, color in COLOR_S08.items():
        a1.bar([], [], color=color, label=z)          # solo para la leyenda
    a1.set(xlabel="capa adaptada (down → mid → up)",
           ylabel=r"$\|\Delta W\|_F\,/\,\|W_0\|_F$",
           title="lora_r8: cuánto se movió cada capa")
    a1.legend()
    resumen = df.pivot_table(index="zona", columns="tipo", values="norma",
                             aggfunc="mean").reindex(list(ZONAS_S08))
    resumen.plot.bar(ax=a2, rot=0, title="promedio por zona y tipo")
    fig.tight_layout()
    plt.show()
    display(resumen.style.format("{:.2e}"))
    print(f"{len(df)} capas adaptadas; la que más se movió: "
          f"{df.loc[df['norma'].idxmax(), 'capa']}")

**➕ Tres lugares para el mismo adaptador** (mismo rango, lr, pasos y semilla):

| Corrida | `target_modules` | Qué cambia |
|---|---|---|
| `lora_r8` | atención en todo el UNet | la referencia de §07 |
| `s08_atencion_mlp` | atención + `ff.net.0.proj`, `ff.net.2` | también el MLP de cada bloque: más parámetros |
| `s08_solo_up` | atención solo en `up_blocks` | solo la mitad que reconstruye: menos parámetros |

- `s08_solo_up` pasa en `modulos` una **regex**, `up_blocks\..*\.(to_q|to_k|to_v|to_out\.0)`: peft la compara con el nombre completo de cada capa, así que solo entran las de `up_blocks`.
- **Qué podría pasar.** Más capas no garantiza menor loss de validation: con pocos pasos las tres curvas pueden quedar casi encimadas. Si `s08_solo_up` queda cerca de `lora_r8`, el estilo vive sobre todo en la parte que reconstruye; si queda claramente arriba, también hace falta mover down y mid.

In [ ]:
ATENCION_S08 = ("to_q", "to_k", "to_v", "to_out.0")
MLP_S08 = ("ff.net.0.proj", "ff.net.2")      # las dos lineales del MLP
# Un solo texto con «(» o «*» la caja lo pasa a peft como REGEX, que debe
# coincidir con el nombre COMPLETO de la capa: atención, solo en up_blocks.
SOLO_UP_S08 = (r"up_blocks\..*\.(to_q|to_k|to_v|to_out\.0)",)

# Todo lo demás es igual a lora_r8 (rango, lr, pasos, schedule, semilla).
VARIANTES_S08 = ["lora_r8", "s08_atencion_mlp", "s08_solo_up"]
if LARGO:
    for c in (Corrida("s08_atencion_mlp", modulos=ATENCION_S08 + MLP_S08),
              Corrida("s08_solo_up", modulos=SOLO_UP_S08)):
        c = entrenar(c)
        print(f"{c.nombre}: " + (c.error or "lista"))
else:
    print("(solo en MODO largo)")

In [ ]:
OK_S08 = [n for n in VARIANTES_S08
          if n in CORRIDAS and CORRIDAS[n].error is None]
if not LARGO:
    print("(solo en MODO largo)")
elif len(OK_S08) < 2:
    print(f"Solo terminó {OK_S08}: no hay nada que comparar.")
else:
    display(tabla(OK_S08))
    curvas(OK_S08, "Dónde poner LoRA: train (delgada) y validation (gruesa)")
    # ‖ΔW‖ relativa promedio por zona: ¿dónde se concentra el cambio?
    zonas = pd.DataFrame({n: por_capa_s08(n).groupby("zona")["norma"].mean()
                          for n in OK_S08}).reindex(list(ZONAS_S08))
    display(zonas.style.format("{:.2e}", na_rep="—"))
    imgs = []
    for n in OK_S08:
        imgs += generar(n, PROMPT, P["semillas"])
    malla(imgs, filas=len(OK_S08),
          titulo=f"«{PROMPT}» · una fila por dónde va LoRA, mismas semillas",
          etiquetas=[f"{n} · s{s}" for n in OK_S08 for s in P["semillas"]])

**Qué mirar**

| Dónde | Pregunta |
|---|---|
| Columna de parámetros entrenables y MB | ¿Cuánto cuesta cada opción? |
| Curvas de validation | ¿Alguna baja más, o son indistinguibles? |
| Tabla por zona | ¿El cambio se concentra en las mismas zonas en las tres corridas? |
| Malla | A igual semilla, ¿cambia el estilo o solo detalles? |

- Si una opción más barata da curva y malla parecidas, es la que conviene: menos parámetros, archivo más chico.

## 09 · Learning rate y schedule

- En el notebook 2 (§08) el learning rate fijaba el tamaño del paso, $\theta\leftarrow\theta-\eta\,\nabla L$; aquí lo nuevo es que $\eta$ **cambia con el paso**: warmup + cosine.
- **Warmup:** $\eta$ sube linealmente de 0 al máximo en los primeros $w$ pasos, para no dar pasos grandes con AdamW aún sin estadísticas ($m$, $v$).
- **Cosine:** después baja suave hasta 0 en el paso final $T$:

$$\eta_t=\eta_{\max}\cdot\tfrac12\Big(1+\cos\big(\pi\,\tfrac{t-w}{T-w}\big)\Big),\qquad w\le t\le T$$

- **Constante:** tras el warmup, $\eta_t=\eta_{\max}$ hasta el final.

**Cómo leer la gráfica.** El learning rate de cada paso para los dos schedules; la línea punteada marca el fin del warmup.

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers.optimization import get_scheduler

R8_S09 = CORRIDAS.get("lora_r8")
PASOS_S09 = (R8_S09.pasos if R8_S09 and R8_S09.pasos else P["pasos"])
# El warmup que usa la caja; si la corrida no lo guardó, ~5 % de los pasos.
WARMUP_S09 = (R8_S09.warmup if R8_S09 and R8_S09.warmup is not None
              else max(1, PASOS_S09 // 20))


def lr_por_paso_s09(nombre_diffusers, lr=1e-4):
    """El learning rate en cada paso, con un optimizer de juguete."""
    opt = torch.optim.AdamW([torch.zeros(1, requires_grad=True)], lr=lr)
    sch = get_scheduler(nombre_diffusers, optimizer=opt,
                        num_warmup_steps=WARMUP_S09,
                        num_training_steps=PASOS_S09)
    valores = []
    for _ in range(PASOS_S09):
        valores.append(sch.get_last_lr()[0])
        opt.step()
        sch.step()
    return valores


fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(lr_por_paso_s09("cosine"), lw=2, label="warmup + cosine (lora_r8)")
ax.plot(lr_por_paso_s09("constant_with_warmup"), lw=2, ls="--",
        label="warmup + constante (s09_constante)")
ax.axvline(WARMUP_S09, color="gray", ls=":", lw=1)
ax.set(xlabel="paso", ylabel="learning rate", title=f"lr máximo 1e-4, "
       f"{PASOS_S09} pasos, warmup de {WARMUP_S09}")
ax.legend()
plt.show()

**➕ Tres corridas nuevas** (todo lo demás igual a `lora_r8`: lr $10^{-4}$, warmup + cosine):

| Corrida | Cambio | Qué se esperaría en la curva | Qué se esperaría en la malla |
|---|---|---|---|
| `s09_lr_1e-5` | lr 10× menor | baja poco y despacio | casi el modelo base |
| `lora_r8` | — | la referencia | el estilo de §07 |
| `s09_lr_1e-3` | lr 10× mayor | baja rápido al inicio; puede saltar o quedar más alta | más estilo, o artefactos y colores saturados |
| `s09_constante` | sin cosine | parecida al inicio; más ruidosa al final | parecida a `lora_r8` |

- Son tendencias, no promesas. LoRA suele tolerar learning rates más altos que el fine-tuning completo ($B$ empieza en 0 y hay pocos parámetros): **si 1e-3 sale bien, no es raro**.
- La loss de difusión depende mucho del $t$ sorteado, así que la curva de train es ruidosa; mira la de validation, que usa $(t,\varepsilon)$ fijos.
- Con pocos pasos, cosine y constante pueden quedar casi iguales: la diferencia está en el tramo final, donde cosine ya casi no mueve los pesos.

In [ ]:
# Todo igual a lora_r8 salvo lo que se cambia.
VARIANTES_S09 = {"s09_lr_1e-5": dict(lr=1e-5),
                 "s09_lr_1e-3": dict(lr=1e-3),
                 "s09_constante": dict(schedule="constante")}
if LARGO:
    for nombre, cambio in VARIANTES_S09.items():
        c = entrenar(Corrida(nombre, **cambio))
        print(f"{nombre}: " + (c.error or "lista"))
else:
    print("(solo en MODO largo) — la tabla de arriba dice qué se esperaría.")

In [ ]:
def listas_s09(nombres):
    """Solo las corridas que existen y terminaron sin error."""
    return [n for n in nombres if n in CORRIDAS and CORRIDAS[n].error is None]


LR_S09 = listas_s09(["s09_lr_1e-5", "lora_r8", "s09_lr_1e-3"])
SCH_S09 = listas_s09(["lora_r8", "s09_constante"])
if not LARGO:
    print("(solo en MODO largo)")
elif len(LR_S09) + len(SCH_S09) < 3:
    print("Faltan corridas: no hay nada que comparar.")
else:
    display(tabla(listas_s09(["lora_r8", *VARIANTES_S09])))
    if len(LR_S09) > 1:
        curvas(LR_S09, "lr 1e-5 / 1e-4 / 1e-3: train (delgada), val (gruesa)")
    if len(SCH_S09) > 1:
        curvas(SCH_S09, "warmup + cosine vs constante (lr 1e-4)")
    filas = LR_S09 + [n for n in SCH_S09 if n not in LR_S09]
    imgs = []
    for n in filas:
        imgs += generar(n, PROMPT, P["semillas"])
    malla(imgs, filas=len(filas),
          titulo=f"«{PROMPT}» · una fila por lr / schedule, mismas semillas",
          etiquetas=[f"{n} · s{s}" for n in filas for s in P["semillas"]])

**Qué mirar**

| Dónde | Pregunta |
|---|---|
| Curvas de lr | ¿Cuál baja más rápido? ¿Alguna sube o salta? |
| Curvas de schedule | ¿Se separan solo al final? |
| Malla | ¿1e-5 se parece al base? ¿1e-3 trae artefactos? |

- Elige con validation y con la malla a la vez: la loss de difusión mide qué tan bien se predice el ruido, no qué tan buena se ve la cara.

## 10 · Cuánto cambia el modelo

- $\Delta W=\frac{\alpha}{r}BA$ tiene rango a lo sumo $r$ porque $A$ tiene $r$ filas y $B$, $r$ columnas.
- La SVD escribe $\Delta W=\sum_i\sigma_i u_i v_i^\top$ con $\sigma_1\ge\sigma_2\ge\dots\ge0$. La **energía** de la dirección $i$ es $\sigma_i^2$; las primeras $k$ direcciones concentran la fracción $E_k=\frac{\sum_{i=1}^k\sigma_i^2}{\lVert\Delta W\rVert_F^2}$, donde $\lVert\Delta W\rVert_F^2=\sum_i\sigma_i^2$.
- **Rango efectivo al 90 %** es el menor $k$ con $E_k\ge0.9$.

**Cómo leer la gráfica.** Cada curva muestra $\sigma_i/\sigma_1$ para una capa, en escala logarítmica. La línea punteada marca $r$. Para $i>r$, los valores caen al nivel del redondeo o al mínimo del eje. En modo largo se repite el análisis con `lora_r64`.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def espectro_s10(nombre):
    """Valores singulares de cada ΔW = (α/r)·B·A sin armar ΔW: con B = Q_B R_B
    y Aᵀ = Q_A R_A (QR reducidas), ΔW = (α/r)·Q_B (R_B R_Aᵀ) Q_Aᵀ, y los
    σ de ΔW son los de la matriz chica r×r R_B R_Aᵀ, por |α/r|."""
    c = CORRIDAS[nombre]
    estado, s = load_file(c.ruta), abs(c.alfa / c.rango)
    salida = {}
    for k in sorted(estado):
        if k.endswith(".lora_A.weight"):
            capa = k[: -len(".lora_A.weight")]
            A = estado[k].float().reshape(estado[k].shape[0], -1)   # [r, d_in]
            B = estado[capa + ".lora_B.weight"].float()             # [d_out, r]
            B = B.reshape(B.shape[0], -1)
            R_B = torch.linalg.qr(B, mode="reduced").R               # [r, r]
            R_A = torch.linalg.qr(A.T, mode="reduced").R             # [r, r]
            salida[capa] = (torch.linalg.svdvals(R_B @ R_A.T) * s).numpy()
    return salida

In [ ]:
def rango_efectivo(s, energia=0.9):
    """El k más chico cuya energía acumulada, con σ², llega a `energia`."""
    acumulada = np.cumsum(s ** 2) / max(float(np.sum(s ** 2)), 1e-30)
    return int(np.searchsorted(acumulada, energia) + 1)


def tres_capas(capas):
    """Una capa de down_blocks, una de mid_block y una de up_blocks."""
    elegidas = []
    for bloque in ("down_blocks", "mid_block", "up_blocks"):
        c = next((c for c in capas if bloque in c and c not in elegidas), None)
        if c is not None:
            elegidas.append(c)
    for c in (capas[0], capas[len(capas) // 2], capas[-1]):  # por si acaso
        if len(elegidas) < 3 and c not in elegidas:
            elegidas.append(c)
    return elegidas


def resumen_s10(nombre):
    """Grafica σ_i/σ_1 de 3 capas e imprime el rango efectivo de todas."""
    c = CORRIDAS.get(nombre)
    if c is None or c.error:
        print(f"{nombre}: no hay adaptador ({c.error if c else 'no entrenado'})")
        return
    esp = espectro_s10(nombre)
    capas = tres_capas(list(esp))
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for capa in capas:
        s = esp[capa][: 2 * c.rango + 4]
        ax.semilogy(np.arange(1, len(s) + 1), np.maximum(s / s[0], 1e-8),
                    "o-", ms=3, label=capa.replace(".processor", "")[-45:])
    ax.axvline(c.rango, color="gray", ls="--", lw=1)
    ax.text(c.rango, 1e-7, f"  r = {c.rango}", color="gray")
    ax.set(xlabel="índice i", ylabel="σ_i / σ_1", ylim=(1e-8, 2),
           title=f"{nombre}: espectro de ΔW en 3 capas")
    ax.legend(fontsize=7)
    plt.show()
    rangos = pd.Series({k: rango_efectivo(s) for k, s in esp.items()})
    display(pd.DataFrame({"σ_1": [esp[k][0] for k in capas],
                          "rango efectivo 90 %": [rangos[k] for k in capas]},
                         index=capas).round(5))
    print(f"{nombre}: {len(rangos)} capas adaptadas, r = {c.rango}; rango "
          f"efectivo mediano {rangos.median():.0f} (mín {rangos.min()}, "
          f"máx {rangos.max()})")


resumen_s10("lora_r8")

In [ ]:
# ➕ El mismo análisis con r = 64: ¿usa el adaptador todo el rango que tiene?
if LARGO:
    resumen_s10("lora_r64")
else:
    print("(solo en MODO largo)")

En inferencia, la perilla $\lambda$ escala el término LoRA en $W=W_0+\lambda\Delta W$.

| $\lambda$ | Actualización aplicada |
|---|---|
| 0 | Base ($W_0$) |
| 0.5 | Mitad de $\Delta W$ |
| 1 | $\Delta W$ tal como se entrenó |
| 1.5 | Una vez y media $\Delta W$ |

**Cómo leer la malla.** Cada fila cambia $\lambda$. Cada columna conserva la semilla y el ruido inicial.

In [ ]:
# Mismas semillas en cada fila: lo único que cambia es λ.
LAMBDAS_S10 = [0.0, 0.5, 1.0, 1.5]
SEM_S10 = P["semillas"][:4]
imgs_s10 = []
for lam in LAMBDAS_S10:
    imgs_s10 += generar("lora_r8", PROMPT, SEM_S10, escala=lam)
malla(imgs_s10, filas=len(LAMBDAS_S10),
      titulo=f"lora_r8 con λ = {LAMBDAS_S10} (filas), «{PROMPT}»",
      etiquetas=[f"λ={lam} · s{s}" for lam in LAMBDAS_S10 for s in SEM_S10])

# λ = 0 contra el modelo base, píxel por píxel (escala 0–255).
base_s10 = generar(None, PROMPT, SEM_S10)
dif_s10 = [np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32))
           for a, b in zip(imgs_s10[: len(SEM_S10)], base_s10)]
print("λ = 0 vs base: diferencia media de píxeles "
      f"{np.mean([d.mean() for d in dif_s10]):.3f}, máxima "
      f"{max(d.max() for d in dif_s10):.0f} (de 255)")

**Qué viste**

- Un rango efectivo muy por debajo de $r$ indica que la energía se concentra en pocas direcciones.
- Para $\lambda=0$, la celda imprime la diferencia absoluta media y máxima de píxeles frente al base. Deberían quedar cerca de cero porque $W_0$ se mantuvo congelada.

## 11 · Memorización

**Memorizar es el overfitting de un modelo generativo:** en vez de aprender la distribución de caras, reproduce caras de train. Buscamos posibles copias. Para cada imagen $g$, medimos su mayor similitud con una cara $x_i$ de train.

$$\text{nn}(g)=\max_i \cos\big(\phi(g),\phi(x_i)\big),\qquad x_i\in\text{train}.$$

| $\phi$ | Un coseno alto indica |
|---|---|
| Embedding de imagen de CLIP | Parecido semántico (tipo de cara, estilo, pose) |
| Píxeles RGB a $64\times64$, centrados por imagen | Copia literal |

- **Referencia.** Las caras reales de validation muestran cuánto se parecen a train otras caras del mismo dominio.
- **Control.** Entrena con menos caras y más pasos que `lora_r8` para favorecer la memorización. Si aún no reproduce caras, sus similitudes pueden quedar cerca de validation.
- **Curvas.** El overfitting del control puede no verse en validation: la loss de difusión es ruidosa porque cada medición mezcla muchos $t$.

In [ ]:
# El control: CONTROL (pocas caras de train), P["pasos_control"] pasos.
CTRL_S11 = entrenar(Corrida("control", datos="control"))
if CTRL_S11.error:
    print(f"control falló: {CTRL_S11.error}")
print(f"control: {len(CONTROL)} caras, {CTRL_S11.pasos or P['pasos_control']} "
      f"pasos; lora_r8: {len(DATOS['train'])} caras, "
      f"{CORRIDAS['lora_r8'].pasos or P['pasos']} pasos")
curvas(["lora_r8", "control"], "train vs validation: lora_r8 y el control")

In [ ]:
import numpy as np
import pandas as pd
import torch


def pixeles_64(imagenes):
    """Vectores de píxeles 64×64 centrados (sin su media) y de norma 1."""
    x = np.stack([np.asarray(im.convert("RGB").resize((64, 64)), np.float32)
                  .ravel() for im in imagenes])
    x -= x.mean(axis=1, keepdims=True)
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)


def vecino(consulta, referencia):
    """Para cada fila de `consulta`: (coseno máximo, índice) en `referencia`."""
    sim = torch.as_tensor(consulta) @ torch.as_tensor(referencia).T
    mejor, idx = sim.max(dim=1)
    return mejor.float().numpy(), idx.numpy()


# Las mismas semillas para los tres modelos; ninguna es de P["semillas"].
N_S11 = 2 if RAPIDO else (16 if LARGO else 8)
SEM_S11 = list(range(100, 100 + N_S11))
GRUPOS_S11 = {"base": generar(None, PROMPT, SEM_S11),
              "lora_r8": generar("lora_r8", PROMPT, SEM_S11),
              "control": generar("control", PROMPT, SEM_S11),
              "validation (reales)": DATOS["validation"]}

TRAIN_CLIP = embeddings_clip(DATOS["train"]).float().cpu()
TRAIN_PIX = pixeles_64(DATOS["train"])
NN_S11 = {}
for grupo, imgs in GRUPOS_S11.items():
    c, ic = vecino(embeddings_clip(imgs).float().cpu(), TRAIN_CLIP)
    p, ip = vecino(pixeles_64(imgs), TRAIN_PIX)
    NN_S11[grupo] = {"clip": c, "idx_clip": ic, "pix": p, "idx_pix": ip}

display(pd.DataFrame({g: {"n": len(v["clip"]),
                          "CLIP mediana": np.median(v["clip"]),
                          "CLIP máx": v["clip"].max(),
                          "píxeles mediana": np.median(v["pix"]),
                          "píxeles máx": v["pix"].max()}
                      for g, v in NN_S11.items()}).T.round(3))

**Cómo leer la gráfica.** Cada punto es una imagen y su altura es el mayor coseno frente a train. Compara `base`, `lora_r8` y `control` con «validation (reales)» en ambos paneles.

In [ ]:
import matplotlib.pyplot as plt

COLOR_S11 = {"base": "#7f7f7f", "lora_r8": "#9467bd", "control": "#d62728",
             "validation (reales)": COLORES["validation"]}
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.8))
rng_s11 = np.random.default_rng(0)
for ax, medida, nombre in zip(ejes, ["clip", "pix"],
                              ["coseno CLIP (parecido)", "coseno de píxeles (copia)"]):
    datos = [NN_S11[g][medida] for g in GRUPOS_S11]
    ax.boxplot(datos, showfliers=False, widths=0.5)
    for i, (g, d) in enumerate(zip(GRUPOS_S11, datos), start=1):
        x = i + rng_s11.uniform(-0.15, 0.15, len(d))
        ax.scatter(x, d, s=12, color=COLOR_S11[g], alpha=0.8, zorder=3)
    ax.set_xticks(range(1, len(GRUPOS_S11) + 1), list(GRUPOS_S11), fontsize=8)
    ax.set(title=f"vecino más cercano en train: {nombre}",
           ylabel="coseno máximo contra train")
plt.tight_layout()
plt.show()

In [ ]:
# Las 4 imágenes del control más parecidas en píxeles a una cara de train,
# arriba; su vecino más cercano en train, abajo.
v = NN_S11["control"]
orden = np.argsort(-v["pix"])[:4]
gen_ctrl = [GRUPOS_S11["control"][i] for i in orden]
vec_ctrl = [DATOS["train"][v["idx_pix"][i]] for i in orden]
malla(gen_ctrl + vec_ctrl, filas=2,
      titulo="control: generada (arriba) ↔ vecino en train (abajo)",
      etiquetas=[f"píx {v['pix'][i]:.2f} · CLIP {v['clip'][i]:.2f}"
                 for i in orden] + ["vecino en train"] * len(orden))

**Qué viste**

- Usa las medianas para comparar grupos y los máximos para detectar posibles copias aisladas.
- Para buscar copias, revisa los pares del control con mayor similitud en píxeles. La imagen generada va arriba y su vecina de train abajo. Si son casi idénticas, hay evidencia de memorización.

## 12 · ¿Olvidó lo que sabía?

`lora_r8` se entrenó con el prompt «an anime face». Comparamos el base y el adaptado con prompts fuera de ese dominio y semillas emparejadas. El **CLIP score** mide la alineación imagen–prompt: $100\times$ el coseno entre sus embeddings.

| Qué ves al activar LoRA | Interpretación |
|---|---|
| El perro sigue siendo reconocible y adquiere estilo anime | Interferencia de estilo |
| El perro deja de ser reconocible | Indicio de pérdida de capacidad con el adaptador activo |

Los efectos pueden ser sutiles si el adaptador cambia poco la salida o si CLIP sigue reconociendo el objeto pese al cambio de estilo. **Interferencia no es olvido:** LoRA no borra nada de $W_0$ y con $\lambda=0$ recuperamos el base (sección 10). Un perro «anime» es interferencia: $\Delta W$ actúa en la atención para cualquier prompt. Solo si el objeto deja de reconocerse con el adaptador activo hablamos de olvido.

**Cómo leer la malla.** Para cada prompt, el base va arriba y `lora_r8` abajo. Cada columna conserva la semilla.

In [ ]:
import pandas as pd

PROMPTS_S12 = ["a photo of a dog", "a city street at night", "a bowl of fruit"]
SEM_S12 = P["semillas"][:3]
# Una imagen por (prompt, semilla), en orden prompt-mayor; misma semilla en
# las dos versiones: lo único que cambia es el adaptador.
base_s12 = generar(None, PROMPTS_S12, SEM_S12)
lora_s12 = generar("lora_r8", PROMPTS_S12, SEM_S12)

n = len(SEM_S12)
imgs, etiquetas, filas = [], [], []
for j, prompt in enumerate(PROMPTS_S12):
    b, l = base_s12[j * n:(j + 1) * n], lora_s12[j * n:(j + 1) * n]
    imgs += b + l
    etiquetas += [f"base · {prompt[:18]}"] * n + [f"lora_r8 · {prompt[:18]}"] * n
    filas.append({"prompt": prompt, "base": clip_score(b, prompt),
                  "lora_r8": clip_score(l, prompt)})
malla(imgs, filas=2 * len(PROMPTS_S12),
      titulo="fuera del dominio: base (fila de arriba) vs lora_r8 (abajo)",
      etiquetas=etiquetas)

# Referencia: el prompt del dominio, con las mismas semillas.
filas.append({"prompt": f"{PROMPT} (dominio)",
              "base": clip_score(generar(None, PROMPT, SEM_S12), PROMPT),
              "lora_r8": clip_score(generar("lora_r8", PROMPT, SEM_S12), PROMPT)})
tabla_s12 = pd.DataFrame(filas).set_index("prompt")
tabla_s12["lora_r8 − base"] = tabla_s12["lora_r8"] - tabla_s12["base"]
display(tabla_s12.round(3))

**Qué viste**

- Una diferencia `lora_r8 − base` negativa indica menor alineación según CLIP. Revisa la malla para ver si se pierde el objeto pedido o cambia su estilo.
- Con pocas semillas, las diferencias pequeñas pueden variar entre muestras. La fila «(dominio)» muestra la comparación de scores con el prompt de entrenamiento.

## 13 · Cierre

- **Test, una vez:** la loss de difusión de LoRA $r=8$ sobre los latents de test, el método que se eligió con validation.
- La tabla de todas las corridas, una receta práctica de LoRA y la tarea.

In [ ]:
# Test, una sola vez, con el método elegido por validation: LoRA r = 8.
# La caja no guarda el UNet entrenado: lo reconstruimos con sus propias
# piezas (base congelado + el adaptador que entrenar() dejó en disco) y
# medimos la misma loss de difusión de validation, ahora sobre test.
from peft import set_peft_model_state_dict
from safetensors.torch import load_file


def unet_con_adaptador(c):
    """UNet base congelado + el adaptador LoRA de la corrida `c`."""
    unet = _cargar_unet(c.metodo)
    unet.add_adapter(_config_lora(c))
    set_peft_model_state_dict(unet, load_file(c.ruta))
    return unet.eval()


elegida = CORRIDAS.get("lora_r8")
if elegida is None or elegida.error or not elegida.ruta:
    print("No hay una corrida \"lora_r8\" sana: corre §7 antes del cierre.")
else:
    liberar()
    unet_test = unet_con_adaptador(elegida)
    with torch.no_grad():
        loss_test = perdida_val(unet_test, LATENTS["test"])
        unet_test.disable_adapters()        # el mismo UNet, sin LoRA
        loss_test_base = perdida_val(unet_test, LATENTS["test"])
    del unet_test
    liberar()
    loss_val = elegida.perdidas_val[-1][1]
    print(f"loss de validation final (lora_r8): {loss_val:.4f}")
    print(f"loss de TEST (lora_r8):             {loss_test:.4f}")
    print(f"loss de test del modelo base:       {loss_test_base:.4f}")
    # Semillas que ninguna sección usó: imágenes nuevas del modelo elegido.
    semillas_test = [100, 101] if RAPIDO else [100, 101, 102, 103]
    malla(generar("lora_r8", semillas=semillas_test), filas=1,
          titulo="lora_r8 · semillas nuevas (test)",
          etiquetas=[f"semilla {s}" for s in semillas_test])

**Cómo leer el resultado**

- Si la loss de test queda cerca de la de validation, validation fue un buen juez para elegir.
- Comparar con el base dice cuánto del estilo se aprendió en datos que nadie miró.
- Las diferencias de loss de difusión suelen ser chicas: léelas contra la de validation y la del base, no en absoluto.
- Si test sale **peor** que el base o muy lejos de validation, es un resultado raro: coméntalo (pocos datos, partición desafortunada, pocos pasos de ruido $K$). No se corrige mirando test otra vez.

- Una fila por corrida: método, rango, módulos, learning rate, pasos, loss de validation final, memoria pico, segundos por paso, MB del adaptador, parámetros entrenables y error.
- Compara **loss de validation contra memoria y tamaño**: ese es el intercambio de LoRA.

In [ ]:
# Todas las corridas registradas, una fila por corrida. Se comparan con la
# loss de validation; las que fallaron (OOM, bitsandbytes) dicen por qué.
resumen = tabla()
if resumen.empty:
    print("CORRIDAS está vacío: ejecuta el notebook completo y vuelve aquí.")
else:
    print(f"{len(resumen)} corridas registradas en MODO {MODO!r}.")
    display(resumen)

**Receta práctica de LoRA (difusión)**

| Perilla | Por dónde empezar | Cuándo moverla |
|---|---|---|
| Rango $r$ | 4–16 | Sube si validation no mejora y el estilo no aparece; baja si memoriza |
| $\alpha$ | $\alpha=r$ (escala 1) | Si cambias $r$, mantener $\alpha/r$ fijo ayuda a no reajustar el learning rate |
| Módulos | Atención: `to_q, to_k, to_v, to_out.0` | Agrega el MLP (`ff.net.*`) si falta capacidad (§8) |
| Learning rate | $10^{-4}$ con AdamW | $10^{-3}$ puede romper la imagen; $10^{-5}$ casi no mueve nada (§9) |
| Schedule | Warmup corto + cosine | Constante sirve para corridas cortas |
| Pasos | Lo que diga la loss de validation y la malla | Para antes de que las caras se vuelvan copias de train (§11) |
| Precisión | LoRA sobre $W_0$ en fp16 | QLoRA solo si no cabe: ahorra memoria, no tiempo |
| Escala $\lambda$ | 1 | $<1$ suaviza el estilo, $>1$ lo exagera (§10) |

- Siempre: semillas fijas para comparar mallas, prompt genérico, y test una vez al final.

**Tarea**

1. Corre el notebook en `MODO = "largo"` y comenta qué cambió en las conclusiones con rangos, módulos y learning rates extra.
2. **DreamBooth:** un token raro (`"a sks face"`) más regularización con imágenes del base. ¿Qué cambia frente al prompt genérico?
3. **Textual inversion:** entrena solo un embedding nuevo, sin tocar el UNet. Compara parámetros y resultados con LoRA.
4. **Colorear bocetos:** plantea cómo condicionar el modelo con el boceto de una cara (img2img o ControlNet) para que la coloree.
5. **SDXL:** repite LoRA $r=8$ y compara memoria y tiempo con SD 1.5.
6. **QLoRA en un LLM:** ajusta un modelo de lenguaje chico con QLoRA; la loss es la cross-entropy del notebook 2.